# 실측 결합형 적응 용접 시뮬레이션: 실시간 최적 조건 탐색 대시보드

## 1. 연구 개요

본 노트북은 저항 점용접(RSW, Resistance Spot Welding) 공정을 대상으로, **실측 데이터에서 학습한
물리 모델**과 **합성 외란(Disturbance) 시뮬레이션**을 결합하여, 공정 중 발생하는 변동에 대해
제어기가 **실시간으로 최적 조건을 탐색하고 대응**하는 폐루프(Closed-loop) 시스템을 구현한다.

### 1.1 선행 연구와의 관계

| 선행 노트북 | 기여 | 한계 |
|---|---|---|
| `1_rsw_optimization.ipynb` | 실측 493건에 물리 모델을 피팅하여 표준 조건 도출 | 표준 조건을 고정 적용, 변동 대응 없음 |
| `3_rsw_adaptive_finetuning.ipynb` | 실측 데이터에 합성 편차를 주입해 적응 제어 비교 | 열입력 축만 보정, 팽출 위험은 미제어 |
| `4_sim.ipynb` | 3x3 디지털 트윈 대시보드로 폐루프 제어 시각화 | 조작 변수가 토치 속도 단일 축, 결함 확률이 수렴하지 않음 |

본 노트북은 위 세 연구를 통합하되, 선행 연구가 공통으로 남긴 **단일 축 제어의 한계**를
다변수(Multi-variable) 최적화로 해결하는 것을 목표로 한다.

### 1.2 핵심 착안점

실측 493건을 조사한 결과, 두 결함 유형이 서로 다른 물리 인자에 지배됨을 확인하였다.

| 결함 유형 | 지배 인자 | 근거 |
|---|---|---|
| 미융착(Bad) | 열입력 $Q$ | $Q$가 임계값 미만일 때 급증하는 로지스틱 관계 |
| 팽출(Explode) | 전극각도, 가압력 | 열입력과는 무관(사분위별 10/5/12/4건, 추세 없음) |

특히 **가압력에 따른 결함률**은 아래와 같이 명확한 최적점을 보인다.

| 가압력 | Explode | Bad | 표본수 |
|---|---|---|---|
| 35 psi | 18.8% | 50.0% | 16 |
| 60 psi | 7.3% | 1.2% | 330 |
| **80 psi** | **1.5%** | **0.0%** | **130** |
| 95 psi | 11.8% | 52.9% | 17 |

즉 가압력이 지나치게 낮으면 전극-모재 접촉이 불충분하여 스패터가 발생하고, 지나치게 높으면
과도한 압입으로 품질이 저하되는 **U자형 관계**가 존재한다. 본 연구는 이 관계를 제2 제어축으로
활용하여, 열입력 축(전류·통전시간)과 팽출 억제 축(가압력)을 동시에 최적화한다.

### 1.3 데이터 구성

본 노트북은 실측과 합성을 명확히 구분하여 결합하는 하이브리드 구조를 취한다.

| 구분 | 변수 | 성격 |
|---|---|---|
| 실측 | 전류, 통전시간, 가압력, 전극각도, 너겟지름, 인장강도, 결함라벨 | 원본 계측 데이터(493건) |
| 실측 | IR 열화상 이미지 99장 | 원본 촬영 이미지 |
| 합성 | 표면온도 편차, 굴곡 편차 | 물리적으로 타당하게 가정한 확률적 외란 |
| 합성 | 센서 관측값 | 위 합성 편차에 측정 노이즈를 더한 값 |

**출처:** 본 시뮬레이션 환경 구축에 활용된 실제 공정 파라미터(전류, 통전시간, 가압력 등) 및
실측 열화상(IR) 이미지 데이터는 Kaggle에 공개된 'Resistance Spot Welding Insights: A Dataset
Integrating Process Parameters, Infrared, and Surface Imaging' 데이터셋을 출처로 한다.

**주의:** 표면온도와 판재 굴곡은 원본 데이터셋에 계측되어 있지 않으므로 합성값을 사용한다.
따라서 본 연구의 결과는 실측 성능 검증이 아니라, 제어 방법론의 타당성을 확인하는
개념 증명(Proof of Concept)으로 해석해야 한다.

In [ ]:
# ==========================================
# [Cell 1] 계산 환경 및 전역 상수 설정
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from matplotlib.colors import ListedColormap
from scipy.optimize import curve_fit
from scipy.stats import linregress
from IPython.display import HTML, display

# 한글 표기 및 음수 부호 렌더링 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 입력 경로: 선행 노트북이 검증해 저장한 실측 집계 결과를 재사용한다
REAL_DATA_CSV = os.path.join("result_rsw", "step1_aggregated_samples.csv")
BINNED_CSV = os.path.join("result_rsw", "step2_binned_stats.csv")
IR_IMAGE_DIR = os.path.join("Data", "Resistance Spot Welding Insights", "ir_images")

# 출력 경로
RESULT_DIR = "result_sim_weld"
FIGURE_DIR = os.path.join(RESULT_DIR, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

# 물리 상수
# ALPHA_R_PER_C: 철강 계열의 전형적인 저항 온도계수 근사값이며,
#                이 데이터셋에서 측정된 값이 아니라 문헌 통용값을 인용한 가정이다.
ALPHA_R_PER_C = 0.004
T_REFERENCE_C = 25.0        # 기준 표면온도 (섭씨)

# 재현성 확보를 위한 난수 시드
SEED = 2026

# 정규화 지수의 안전/경계/위험 구간 경계 (단위: %)
SAFE_LO, SAFE_HI = 90.0, 110.0
CAUTION_LO, CAUTION_HI = 75.0, 125.0

# 대시보드 공통 색상 (배경 채움 / 전경 표식)
C_SAFE, C_CAUTION, C_RISK = "#d9f0d3", "#fdf0c3", "#f4cbcb"
FG_SAFE, FG_CAUTION, FG_RISK = "#2ca02c", "#ff9800", "#d62728"
C_BASELINE, C_OPTIMAL = "#7f7f7f", "#1f77b4"
C_BASELINE_NUGGET = "#ff7f0e"  # 1행2열 단면도에서 optimal(파랑)과 뚜렷이 구분되는 baseline 색상

print("[초기화] 결과 저장 경로:", RESULT_DIR)

## 2. 실측 데이터 적재 및 물리 모델 피팅

### 2.1 데이터 저장소

`WeldDataRepository`는 실측 집계 데이터(493건), 구간별 통계, IR 열화상 이미지 목록을 적재한다.
애니메이션은 IR 이미지를 보유한 99건을 열입력 오름차순으로 정렬하여 프레임 단위로 재생하므로,
**모든 프레임이 실제로 수행된 용접 1건에 대응**한다.

### 2.2 피팅 대상 물리 모델

실측 493건으로부터 다음 네 가지 모델을 피팅한다. 이 모델들은 이후 최적 조건 탐색의 목적함수를
구성하는 근거가 된다.

**(1) 미융착(Bad) 확률 - 감소형 로지스틱**

$$p_{bad}(Q)=\frac{1}{1+e^{k(Q-Q_{min})}}$$

* $p_{bad}(Q)$: 열입력 $Q$에서 예측되는 미융착 확률(0~1).
* $Q$: 열입력 지표(전류의 제곱과 통전시간의 곱, $I^2 t$).
* $Q_{min}$: 미융착 확률이 정확히 0.5가 되는 임계 열입력. 실측 데이터로부터 피팅한다.
* $k$: 로지스틱 곡선의 형상 계수. 값이 클수록 임계값 부근에서 확률이 급격하게 변한다.
* $e$: 자연상수.

열입력 $Q$가 임계값 $Q_{min}$보다 낮을수록 모재가 충분히 융해되지 못해 미융착 확률이 1에
접근한다. $Q=Q_{min}$에서 확률은 정확히 0.5가 된다.

**(2) 팽출(Explode) 확률 - 각도 및 가압력의 2변수 로지스틱**

$$p_{exp}(\theta,P)=\frac{1}{1+e^{-z}},\qquad z=b_0+b_\theta\,\theta+b_1 P+b_2 P^2$$

* $p_{exp}(\theta,P)$: 접촉각 $\theta$와 가압력 $P$에서 예측되는 팽출 확률.
* $\theta$: 유효 접촉각(도).
* $P$: 가압력(psi).
* $z$: 로지스틱 함수에 대입되는 선형 결합 값(로짓, logit).
* $b_0$: 절편 계수.
* $b_\theta$: 접촉각의 영향을 나타내는 계수.
* $b_1$, $b_2$: 가압력의 1차항과 2차항 계수. $b_2>0$이면 가압력에 대해 U자형 관계가 나타난다.

전극각도 $\theta$가 클수록 접촉이 기울어져 스패터 위험이 증가한다. 가압력 $P$에 대해서는
1차항과 2차항을 함께 두어, 1.2절에서 확인한 **U자형 관계**(과소 가압 시 접촉 불량, 과대 가압 시
품질 저하)를 표현할 수 있도록 한다. 이 2차항이 실시간 최적 가압력을 탐색 가능하게 만드는
핵심 장치이다.

**(3) 너겟 성장 - 포화형 지수 곡선**

$$D(Q)=D_0+(D_{max}-D_0)\left(1-e^{-Q/\tau}\right)$$

* $D(Q)$: 열입력 $Q$에서 예측되는 너겟 지름(mm).
* $D_0$: 열입력이 0에 가까울 때의 초기 지름.
* $D_{max}$: 열입력이 충분히 클 때 수렴하는 포화 지름.
* $\tau$: 성장 속도를 결정하는 시상수. 값이 작을수록 빠르게 포화값에 도달한다.

열입력이 증가하면 용융 금속량이 늘어나지만, 판재 두께와 전극 냉각의 제약으로 최대 지름
$D_{max}$에 점근한다.

**(4) 인장 전단 강도 - 선형 회귀**

$$F=a\,D+b$$

* $F$: 예측 인장 전단 강도(N).
* $D$: 너겟 지름(mm).
* $a$: 회귀 기울기(너겟 지름이 1mm 늘어날 때 강도 증가량).
* $b$: 회귀 절편.

너겟 단면적이 클수록 파단에 필요한 힘이 증가한다는 관계를 실측값으로 근사한다.

In [ ]:
# ==========================================
# [Cell 2] 데이터 저장소 및 물리 모델 피팅기
# ==========================================

def bad_probability_model(Q, Q_min, k):
    """미융착(Bad) 확률: 열입력이 임계값 Q_min 아래로 갈수록 1에 접근하는 감소형 로지스틱."""
    return 1.0 / (1.0 + np.exp(k * (Q - Q_min)))


def explode_probability_model(X, b0, b_angle, b_p1, b_p2):
    """팽출(Explode) 확률: 전극각도와 가압력을 설명변수로 하는 로지스틱.

    가압력에 1차항과 2차항을 모두 포함하여, 과소 가압과 과대 가압 양쪽에서
    위험이 증가하는 U자형 관계를 표현할 수 있도록 한다.
    """
    angle, pressure = X
    z = b0 + b_angle * angle + b_p1 * pressure + b_p2 * pressure ** 2
    return 1.0 / (1.0 + np.exp(-z))


def nugget_growth_model(Q, D0, Dmax, tau):
    """너겟 성장: 열입력 증가에 따라 최대 지름 Dmax로 포화하는 지수 곡선."""
    return D0 + (Dmax - D0) * (1.0 - np.exp(-Q / tau))


class WeldDataRepository:
    """실측 저항 점용접 데이터와 IR 열화상 이미지를 적재한다."""

    def __init__(self, data_csv, binned_csv, image_dir):
        self.samples = pd.read_csv(data_csv)
        self.binned = pd.read_csv(binned_csv)
        self.image_dir = image_dir
        self.image_ids = self._scan_image_ids()

    def _scan_image_ids(self):
        """IR_<번호>.jpg 형태의 파일명에서 시료 번호를 추출한다."""
        ids = []
        for name in os.listdir(self.image_dir):
            if name.startswith("IR_") and name.lower().endswith(".jpg"):
                ids.append(int(name.split("_")[1].split(".")[0]))
        return sorted(ids)

    def demo_frames(self):
        """IR 이미지를 보유한 시료만 열입력 오름차순으로 정렬하여 반환한다.

        각 행이 애니메이션의 한 프레임에 대응하므로, 모든 프레임은
        실제로 수행된 용접 1건을 근거로 삼는다.
        """
        subset = self.samples[self.samples["sample_id"].isin(self.image_ids)]
        return subset.sort_values("heat_input_proxy").reset_index(drop=True)

    def image_path(self, sample_id):
        return os.path.join(self.image_dir, "IR_%d.jpg" % int(sample_id))


class ProcessModelFitter:
    """실측 데이터로부터 결함 확률과 품질 지표의 물리 모델을 피팅한다."""

    def __init__(self, samples):
        self.samples = samples
        self.params = {}

    def fit_all(self):
        self._fit_bad()
        self._fit_explode()
        self._fit_nugget()
        self._fit_tensile()
        return self.params

    def _fit_bad(self):
        """열입력에 대한 미융착 확률 로지스틱을 피팅한다."""
        Q = self.samples["heat_input_proxy"].values
        y = self.samples["is_bad"].values.astype(float)
        span = Q.max() - Q.min()
        # 초기값과 탐색 범위를 물리적으로 타당한 구간으로 제한하여 비물리적 해를 차단한다
        p0 = [np.percentile(Q, 15), 10.0 / span]
        bounds = ([Q.min(), 1e-8], [Q.max(), 50.0 / span])
        popt, pcov = curve_fit(bad_probability_model, Q, y, p0=p0,
                               bounds=bounds, maxfev=20000)
        err = np.sqrt(np.diag(pcov))
        self.params["Q_min"] = popt[0]
        self.params["k_bad"] = popt[1]
        self.params["Q_min_err"] = err[0]

    def _fit_explode(self):
        """전극각도와 가압력에 대한 팽출 확률 로지스틱을 피팅한다."""
        angle = self.samples["angle_deg"].values.astype(float)
        pressure = self.samples["pressure_psi"].values.astype(float)
        y = self.samples["is_explode"].values.astype(float)
        p0 = [0.0, 0.09, -0.05, 0.0003]
        popt, _ = curve_fit(explode_probability_model, (angle, pressure), y,
                            p0=p0, maxfev=40000)
        self.params["b0_exp"] = popt[0]
        self.params["b_angle"] = popt[1]
        self.params["b_p1"] = popt[2]
        self.params["b_p2"] = popt[3]

    def _fit_nugget(self):
        """열입력에 대한 너겟 지름 성장 곡선을 피팅한다."""
        Q = self.samples["heat_input_proxy"].values
        D = self.samples["nugget_diameter_mm"].values
        p0 = [D.min(), D.max(), np.median(Q)]
        popt, _ = curve_fit(nugget_growth_model, Q, D, p0=p0, maxfev=20000)
        self.params["D0"] = popt[0]
        self.params["Dmax"] = popt[1]
        self.params["tau"] = popt[2]

    def _fit_tensile(self):
        """너겟 지름에 대한 인장 전단 강도 선형 회귀를 수행한다."""
        lr = linregress(self.samples["nugget_diameter_mm"],
                        self.samples["pull_test_N"])
        self.params["F_slope"] = lr.slope
        self.params["F_intercept"] = lr.intercept
        self.params["F_rvalue"] = lr.rvalue


# --- 실행 ---------------------------------------------------------
repo = WeldDataRepository(REAL_DATA_CSV, BINNED_CSV, IR_IMAGE_DIR)
frames = repo.demo_frames()
params = ProcessModelFitter(repo.samples).fit_all()

# 피팅 결과를 중간 산출물로 저장한다
pd.DataFrame([params]).to_csv(
    os.path.join(RESULT_DIR, "step1_fitted_models.csv"), index=False)

# 팽출 모델의 근거가 되는 각도-가압력 교차 집계도 함께 저장한다
cross = (repo.samples.groupby(["angle_deg", "pressure_psi"])
         .agg(n=("is_explode", "size"),
              n_explode=("is_explode", "sum"),
              n_bad=("is_bad", "sum")).reset_index())
cross["explode_rate"] = cross["n_explode"] / cross["n"]
cross["bad_rate"] = cross["n_bad"] / cross["n"]
cross.to_csv(os.path.join(RESULT_DIR, "step2_explode_pressure_stats.csv"),
             index=False)

print("[적재] 실측 %d건, IR 이미지 %d장, 재생 프레임 %d개"
      % (len(repo.samples), len(repo.image_ids), len(frames)))
print("[피팅] Q_min = %.3e +/- %.3e,  k_bad = %.3e"
      % (params["Q_min"], params["Q_min_err"], params["k_bad"]))
print("[피팅] 팽출 계수: 각도 %+.4f, 가압력 1차 %+.4f, 가압력 2차 %+.6f"
      % (params["b_angle"], params["b_p1"], params["b_p2"]))
print("[피팅] 너겟 Dmax = %.3f mm,  tau = %.3e" % (params["Dmax"], params["tau"]))
print("[피팅] 인장강도 F = %.1f D %+.1f  (r = %.3f)"
      % (params["F_slope"], params["F_intercept"], params["F_rvalue"]))

## 3. 공정 외란 모델링

### 3.1 외란의 물리적 의미

실제 생산 현장에서는 모든 용접이 동일한 표준 조건에서 이루어지지 않는다. 앞 공정의 열이 남아
모재 표면이 예열되어 있거나, 판재가 미세하게 휘어 전극과의 접촉각이 설계값에서 벗어나는 상황이
빈번하게 발생한다. 본 절에서는 이러한 변동을 두 가지 외란으로 모델링한다.

**(1) 표면온도 편차 $\Delta T$ - 접촉저항 변화**

금속의 전기저항은 온도가 상승하면 함께 증가한다. 이를 선형 근사하면 유효 접촉저항은 다음과 같다.

**주의(용융풀 온도와의 구분):** 여기서 다루는 표면온도 편차 $\Delta T$는 아크에 의해 순간적으로
1,500~1,800도까지 올라가는 **용융풀(Weld Pool) 온도가 아니라**, 용접이 시작되기 전 모재 자체가
앞선 공정의 잔열이나 주변 환경 때문에 상온(25도) 기준에서 벗어나 있는 정도를 의미한다. 따라서
그 크기는 수십 도 이내로 가정하며, 이는 `3_rsw_adaptive_finetuning.ipynb`의 `sim_temp_dev_C`와
동일한 개념이다. 용접 중 실제 용융풀의 열 이력을 별도로 모사하지는 않는다.

$$R_{eff}=1+\alpha\,\Delta T,\qquad \alpha=0.004\ [1/^\circ\mathrm{C}]$$

* $R_{eff}$: 표준 조건(=1)을 기준으로 한 유효 접촉저항의 상대값.
* $\Delta T$: 표준 조건 대비 표면온도 편차(섭씨).
* $\alpha$: 저항 온도계수. 철강 계열의 전형적 근사값이며 이 데이터셋에서 측정된 값은 아니다.

따라서 동일한 전류와 통전시간을 인가하더라도 실제 전달되는 열입력이 달라진다.

$$Q_{eff}=I^2 R_{eff}\, t$$

* $Q_{eff}$: 온도 외란이 반영된 유효 열입력.
* $I$: 전류(A).
* $R_{eff}$: 위에서 정의한 유효 접촉저항.
* $t$: 통전시간(s).

여기서 $\Delta T$는 절대 온도가 아니라 **표준 조건 대비 상대 편차**이다. 표준 조건이란 실측
데이터가 수집될 당시의 (계측되지 않은) 암묵적 온도 상태를 의미하며, $R_{eff}$가 곱셈 형태로만
사용되므로 절대 기준값을 정할 필요가 없다.

**(2) 굴곡 편차 $\Delta\theta$ - 유효 접촉각 변화**

판재의 곡률로 인해 실제 접촉각은 설계 전극각도에서 추가로 벗어난다.

$$\theta_{eff}=\theta_{설계}+\Delta\theta$$

* $\theta_{eff}$: 실제로 작용하는 유효 접촉각.
* $\theta_{설계}$: 설계 전극각도(0도 또는 15도).
* $\Delta\theta$: 판재 굴곡으로 인한 접촉각 편차.

### 3.2 확률 과정의 형태

두 편차는 모두 **평균 0의 가우시안 증분이 누적되는 랜덤워크**로 생성한다.

$$\Delta T_i=\sum_{j\le i}\varepsilon_j,\qquad \varepsilon_j\sim N(0,\sigma^2)$$

* $\Delta T_i$: $i$번째 프레임까지 누적된 온도 편차.
* $i, j$: 프레임(시간 스텝)을 나타내는 인덱스.
* $\varepsilon_j$: $j$번째 스텝에서 새로 발생한 확률적 증분.
* $N(0,\sigma^2)$: 평균 0, 분산 $\sigma^2$인 정규분포.

랜덤워크를 선택한 이유는, 공정이 진행됨에 따라 표준 조건에서 **점진적으로 이탈**하는 실제 현상
(누적 입열에 의한 모재 승온, 전극 마모에 따른 자세 변화)을 표현하기 위함이다. 이 확률 과정은
시간에 따라 분산이 증가하는 비정상(Non-stationary) 과정이므로, 공정 후반으로 갈수록 제어기의
개입 필요성이 커진다.

### 3.3 센서 관측 모델

제어기가 편차의 참값을 직접 알 수 있다면 비교 실험의 의미가 없다. 따라서 제어기에는 측정
노이즈가 더해진 관측값만을 제공한다.

$$\Delta T_{obs}=\Delta T+\nu_T,\qquad \nu_T\sim N(0,\sigma_T^2)$$

* $\Delta T_{obs}$: 센서가 관측하는 온도 편차. 제어기가 실제로 참조하는 값이다.
* $\Delta T$: 편차 참값.
* $\nu_T$: 측정 노이즈.
* $\sigma_T^2$: 측정 노이즈의 분산.

즉 제어기는 **불완전한 정보로 판단**하며, 성능 평가는 참값 기준으로 이루어진다.

In [ ]:
# ==========================================
# [Cell 3] 공정 외란 생성기
# ==========================================

class DisturbanceGenerator:
    """표준 조건을 벗어난 시공 상황을 확률적으로 합성한다.

    실측 데이터셋에는 표면온도와 판재 굴곡이 계측되어 있지 않으므로,
    물리적으로 타당한 범위 안에서 누적 랜덤워크로 생성한다.
    생성값은 모두 합성(SIMULATED)값이며 실측이 아니다.
    """

    # 랜덤워크 1스텝당 표준편차 (누적되어 공정 후반에 큰 편차를 형성한다)
    TEMP_DRIFT_STD = 1.5      # 섭씨
    CURV_DRIFT_STD = 0.4      # 도

    # 센서 측정 노이즈 표준편차
    TEMP_SENSOR_STD = 3.0     # 섭씨
    CURV_SENSOR_STD = 1.5     # 도

    def __init__(self, n_frames, seed=SEED):
        self.n_frames = n_frames
        self.rng = np.random.default_rng(seed)

    def generate(self):
        n = self.n_frames
        # 참값: 누적 랜덤워크로 표준 조건에서 점진적으로 이탈한다
        temp_true = np.cumsum(self.rng.normal(0.0, self.TEMP_DRIFT_STD, n))
        curv_true = np.cumsum(self.rng.normal(0.0, self.CURV_DRIFT_STD, n))

        # 관측값: 참값에 측정 노이즈가 더해진 값으로, 제어기는 이 값만 참조한다
        temp_obs = temp_true + self.rng.normal(0.0, self.TEMP_SENSOR_STD, n)
        curv_obs = curv_true + self.rng.normal(0.0, self.CURV_SENSOR_STD, n)

        return pd.DataFrame({
            "frame": np.arange(n),
            "temp_dev_true": temp_true,
            "curv_dev_true": curv_true,
            "temp_dev_obs": temp_obs,
            "curv_dev_obs": curv_obs,
        })


# --- 실행 ---------------------------------------------------------
disturbance = DisturbanceGenerator(len(frames)).generate()
disturbance.to_csv(os.path.join(RESULT_DIR, "step3_disturbance.csv"), index=False)

print("[외란] 표면온도 편차 범위: %+.1f ~ %+.1f 섭씨"
      % (disturbance["temp_dev_true"].min(), disturbance["temp_dev_true"].max()))
print("[외란] 굴곡 편차 범위: %+.1f ~ %+.1f 도"
      % (disturbance["curv_dev_true"].min(), disturbance["curv_dev_true"].max()))

## 4. 실시간 최적 조건 탐색

### 4.1 제어 구조

본 제어기는 매 프레임마다 센서 관측값으로 외란을 추정한 뒤, **격자 탐색(Grid Search)**으로
목적함수를 최소화하는 조작 변수 조합을 산출한다. 조작 변수는 세 가지이며 각각 서로 다른
결함 축을 담당한다.

| 조작 변수 | 기호 | 담당 결함 축 | 물리적 경로 |
|---|---|---|---|
| 전류 | $I$ | 미융착 | $Q=I^2R_{eff}t$ 를 통해 열입력 조절 |
| 통전시간 | $t$ | 미융착 | 동일 |
| 가압력 | $P$ | 팽출 | 전극-모재 접촉 상태를 통해 스패터 억제 |

선행 연구가 열입력 축 하나만 제어하여 팽출 위험을 방치했던 것과 달리, 본 구조는 두 결함 축을
**독립적인 조작 변수로 동시에 제어**한다.

### 4.2 목적함수

$$J(I,t,P)=w_{bad}\,p_{bad}(Q)+w_{exp}\,p_{exp}(\theta_{eff},P)+w_{heat}\left(\frac{Q-Q_{target}}{Q_{target}}\right)^2$$

* $J(I,t,P)$: 조작 변수 조합 $(I,t,P)$에서 산출되는 목적함수 값. 값이 작을수록 좋은 조건이다.
* $I,t,P$: 각각 전류, 통전시간, 가압력(조작 변수).
* $w_{bad},w_{exp},w_{heat}$: 세 항의 상대적 중요도를 조절하는 가중치.
* $p_{bad}(Q)$: 미융착 확률(2.2절 (1) 참고).
* $p_{exp}(\theta_{eff},P)$: 팽출 확률(2.2절 (2) 참고).
* $Q$: 조작 변수 $(I,t)$로부터 계산되는 열입력.
* $Q_{target}$: 목표 열입력.

세 항의 역할은 다음과 같다.

1. **미융착 위험항**: 열입력이 부족할 때 증가한다.
2. **팽출 위험항**: 접촉각이 크거나 가압력이 최적점에서 벗어날 때 증가한다.
3. **열입력 추종항**: 목표 열입력에서 벗어난 정도를 제곱으로 벌점화한다. 이 항이 없으면
   미융착 위험만 낮추기 위해 열입력을 무한정 높이는 비현실적 해가 선택되므로,
   공정 규격을 유지하는 제약 역할을 한다.

### 4.3 탐색 공간

| 변수 | 범위 | 격자 수 | 근거 |
|---|---|---|---|
| 전류 | 공칭값의 0.85 ~ 1.15배 | 11 | 설비의 현실적 조정 범위 |
| 통전시간 | 0.2 ~ 1.5 초 | 7 | 실측 데이터에 존재하는 이산 수준 |
| 가압력 | 35 ~ 95 psi | 25 | 실측 데이터의 관측 범위 내부 |

가압력은 실측에 4개 수준(35, 60, 80, 95 psi)만 존재하므로, 그 사이 값은 피팅된 2차 로지스틱에
의한 **내삽(Interpolation)**임에 유의해야 한다. 관측 범위를 벗어난 외삽은 수행하지 않는다.

### 4.4 비교 실험 설계

| 운전 방식 | 조작 변수 결정 방법 |
|---|---|
| **baseline** | 실측 원래 설정값을 그대로 사용하며 외란을 인식하지 못한다. 현장의 고정 설정 관행에 해당한다. |
| **optimal** | 매 프레임 센서 관측값으로 외란을 추정하고 격자 탐색으로 조작 변수를 재산정한다. |

두 방식 모두 **성능 평가는 참값 기준**으로 수행한다. 즉 제어기는 노이즈가 섞인 관측값으로
판단하지만, 그 결과로 발생하는 결함 확률은 실제 외란 참값을 대입하여 계산한다.

In [ ]:
# ==========================================
# [Cell 4] 실시간 최적 조건 탐색기 및 폐루프 시뮬레이터
# ==========================================

class OperatingPointOptimizer:
    """관측된 외란 조건에서 결함 확률을 최소화하는 조작 변수를 격자 탐색으로 산출한다.

    전류와 통전시간은 열입력을 통해 미융착 위험을 지배하고,
    가압력은 전극-모재 접촉 상태를 통해 팽출 위험을 지배한다.
    """

    def __init__(self, params, Q_target,
                 current_scale=None, time_grid=None, pressure_grid=None,
                 w_bad=1.0, w_explode=1.0, w_heat=0.5):
        self.params = params
        self.Q_target = Q_target
        # 전류는 공칭값 대비 배율로 탐색하여 시료별 운전점 차이를 흡수한다
        self.current_scale = (np.linspace(0.85, 1.15, 11)
                              if current_scale is None else current_scale)
        # 통전시간은 실측에 존재하는 이산 수준을 그대로 사용한다
        self.time_grid = (np.array([0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5])
                          if time_grid is None else time_grid)
        # 가압력은 실측 관측 범위 내부에서만 탐색한다 (외삽 금지)
        self.pressure_grid = (np.linspace(35.0, 95.0, 25)
                              if pressure_grid is None else pressure_grid)
        self.w_bad = w_bad
        self.w_explode = w_explode
        self.w_heat = w_heat

    def _explode_by_pressure(self, tilt):
        """주어진 접촉각에서 가압력 격자별 팽출 확률을 계산한다."""
        angle_vec = np.full(len(self.pressure_grid), tilt)
        return explode_probability_model(
            (angle_vec, self.pressure_grid),
            self.params["b0_exp"], self.params["b_angle"],
            self.params["b_p1"], self.params["b_p2"])

    def search(self, current_nominal, R_eff_est, tilt_est):
        """단일 시점에서 목적함수를 최소화하는 (전류, 통전시간, 가압력)을 반환한다.

        세 조작 변수의 격자를 3차원으로 전개하여 한 번에 평가한다.
        """
        # 축 배치: (전류, 통전시간, 가압력)
        I = (current_nominal * self.current_scale)[:, None, None]
        t = self.time_grid[None, :, None]

        # 열입력과 그에 따른 미융착 위험 (가압력과 무관하므로 2차원까지만 계산)
        Q = I ** 2 * R_eff_est * t
        p_bad = bad_probability_model(Q, self.params["Q_min"], self.params["k_bad"])

        # 팽출 위험은 가압력에만 의존하므로 1차원 계산 후 방송(broadcast)한다
        p_exp = self._explode_by_pressure(tilt_est)[None, None, :]

        # 열입력 추종 벌점: 목표에서 벗어난 정도를 제곱으로 반영한다
        heat_penalty = ((Q - self.Q_target) / self.Q_target) ** 2

        J = self.w_bad * p_bad + self.w_explode * p_exp + self.w_heat * heat_penalty
        i, j, k = np.unravel_index(np.argmin(J), J.shape)

        return {
            "current": float(current_nominal * self.current_scale[i]),
            "time": float(self.time_grid[j]),
            "pressure": float(self.pressure_grid[k]),
        }


class ClosedLoopSimulator:
    """무보정(baseline) 운전과 최적 조건 탐색(optimal) 운전을 동일 외란 아래에서 비교한다."""

    def __init__(self, frames, disturbance, params, optimizer):
        self.frames = frames
        self.disturbance = disturbance
        self.params = params
        self.optimizer = optimizer

    def _quality(self, Q, tilt, pressure):
        """주어진 운전 조건에서 결함 확률과 품질 지표를 예측한다."""
        p_bad = float(bad_probability_model(
            Q, self.params["Q_min"], self.params["k_bad"]))
        p_exp = float(explode_probability_model(
            (np.array([tilt]), np.array([pressure])),
            self.params["b0_exp"], self.params["b_angle"],
            self.params["b_p1"], self.params["b_p2"])[0])
        nugget = float(nugget_growth_model(
            Q, self.params["D0"], self.params["Dmax"], self.params["tau"]))
        tensile = self.params["F_slope"] * nugget + self.params["F_intercept"]
        # 두 결함이 독립이라 가정할 때, 적어도 하나가 발생할 확률
        p_total = 1.0 - (1.0 - p_bad) * (1.0 - p_exp)
        return p_bad, p_exp, p_total, nugget, float(tensile)

    def run(self):
        records = []
        for i in range(len(self.frames)):
            row = self.frames.iloc[i]
            dist = self.disturbance.iloc[i]

            # 참값: 실제로 작용한 물리 상태 (성능 평가에 사용)
            R_true = 1.0 + ALPHA_R_PER_C * dist["temp_dev_true"]
            tilt_true = row["angle_deg"] + dist["curv_dev_true"]

            # 관측값: 센서가 감지한 값 (제어기 판단에만 사용)
            R_obs = 1.0 + ALPHA_R_PER_C * dist["temp_dev_obs"]
            tilt_obs = row["angle_deg"] + dist["curv_dev_obs"]

            # (1) baseline: 외란을 인식하지 못하고 원래 설정값을 유지한다
            Q_base = row["avg_current_A"] ** 2 * R_true * row["weld_time_s"]
            b_bad, b_exp, b_tot, b_nug, b_ten = self._quality(
                Q_base, tilt_true, row["pressure_psi"])

            # (2) optimal: 관측값으로 최적 조건을 탐색한 뒤 참값으로 결과를 평가한다
            best = self.optimizer.search(row["avg_current_A"], R_obs, tilt_obs)
            Q_opt = best["current"] ** 2 * R_true * best["time"]
            o_bad, o_exp, o_tot, o_nug, o_ten = self._quality(
                Q_opt, tilt_true, best["pressure"])

            records.append({
                "frame": i,
                "sample_id": int(row["sample_id"]),
                "category": row["category"],
                "angle_deg": row["angle_deg"],
                "temp_dev_true": dist["temp_dev_true"],
                "curv_dev_true": dist["curv_dev_true"],
                "tilt_true": tilt_true,
                "R_eff_true": R_true,
                # baseline 운전 결과
                "base_current": row["avg_current_A"],
                "base_time": row["weld_time_s"],
                "base_pressure": row["pressure_psi"],
                "base_Q": Q_base, "base_p_bad": b_bad, "base_p_exp": b_exp,
                "base_p_total": b_tot, "base_nugget": b_nug, "base_tensile": b_ten,
                # optimal 운전 결과
                "opt_current": best["current"],
                "opt_time": best["time"],
                "opt_pressure": best["pressure"],
                "opt_Q": Q_opt, "opt_p_bad": o_bad, "opt_p_exp": o_exp,
                "opt_p_total": o_tot, "opt_nugget": o_nug, "opt_tensile": o_ten,
            })
        return pd.DataFrame(records)


# --- 실행 ---------------------------------------------------------
# 목표 열입력: 실측 493건의 중앙값을 공정 규격 기준점으로 삼는다.
# 이 수준에서는 미융착 확률이 사실상 0에 수렴하므로 안전 여유가 충분하다.
Q_TARGET = float(repo.samples["heat_input_proxy"].median())

optimizer = OperatingPointOptimizer(params, Q_TARGET)
log = ClosedLoopSimulator(frames, disturbance, params, optimizer).run()
log.to_csv(os.path.join(RESULT_DIR, "step4_control_log.csv"), index=False)

# 최종 비교 요약을 별도 파일로 저장한다
summary = pd.DataFrame([{
    "n_frames": len(log),
    "Q_target": Q_TARGET,
    "base_p_bad_mean": log["base_p_bad"].mean(),
    "opt_p_bad_mean": log["opt_p_bad"].mean(),
    "base_p_exp_mean": log["base_p_exp"].mean(),
    "opt_p_exp_mean": log["opt_p_exp"].mean(),
    "base_p_total_mean": log["base_p_total"].mean(),
    "opt_p_total_mean": log["opt_p_total"].mean(),
    "defect_reduction_pct": 100.0 * (1.0 - log["opt_p_total"].mean()
                                     / log["base_p_total"].mean()),
    "base_tensile_mean": log["base_tensile"].mean(),
    "opt_tensile_mean": log["opt_tensile"].mean(),
}])
summary.to_csv(os.path.join(RESULT_DIR, "step5_summary.csv"), index=False)

print("[목표] 열입력 규격 Q_target = %.3e" % Q_TARGET)
print("[결과] 평균 미융착 확률  baseline %.4f -> optimal %.4f"
      % (log["base_p_bad"].mean(), log["opt_p_bad"].mean()))
print("[결과] 평균 팽출 확률    baseline %.4f -> optimal %.4f"
      % (log["base_p_exp"].mean(), log["opt_p_exp"].mean()))
print("[결과] 평균 종합 결함확률 baseline %.4f -> optimal %.4f  (%.1f%% 감소)"
      % (log["base_p_total"].mean(), log["opt_p_total"].mean(),
         summary["defect_reduction_pct"].iloc[0]))
print("[결과] 최적 가압력 중앙값 = %.1f psi" % log["opt_pressure"].median())

## 5. 3x3 디지털 트윈 대시보드 구성

### 5.1 공차 기반 무차원 정규화

온도(섭씨), 각도(도), 지름(mm), 전류(A) 등 서로 다른 단위를 갖는 물리량을 하나의 화면에서
비교하기 위하여, 각 변수 고유의 허용 공차로 나눈 **무차원 정규화 지수**를 도입한다.

$$I_{norm}=100+10\left(\frac{X-X_0}{\Delta X_{safe}}\right)\ [\%]$$

* $I_{norm}$: 정규화 지수(%). 100%가 표준 조건에 해당한다.
* $X$: 실시간으로 측정되는 변수(온도, 각도 등).
* $X_0$: 공정 표준값.
* $\Delta X_{safe}$: 안전성을 유지할 수 있는 허용 공차.

여기서 $X_0$는 공정 표준값, $\Delta X_{safe}$는 해당 변수가 안전성을 유지할 수 있는 허용 공차이다.
정의상 $X=X_0$이면 100%가 되고, 공차 한 단위만큼 벗어날 때마다 10%씩 이동한다.

| 상태 | 정규화 지수 | 색상 | 해석 |
|---|---|---|---|
| 안전 | 90 ~ 110% | 초록 | 공차 1배 이내로 벗어남 |
| 경계 | 75~90% 또는 110~125% | 노랑 | 공차 1~2.5배 사이로 벗어남 |
| 위험 | 75% 미만 또는 125% 초과 | 빨강 | 공차 2.5배를 초과하여 벗어남 |

식의 정의에 따라 지수가 100%에서 10%p 이동할 때마다 $X$는 공차 $\Delta X_{safe}$ 한 단위만큼
이동한 것이므로, 위 표의 경계값(90, 110, 75, 125%)은 각각 공차의 1배와 2.5배 지점에 해당한다.
1행 3열(환경 및 품질 지수)과 2행 3열(장비 변수 지수)에 사용되는 변수별 공차 $\Delta X_{safe}$는
다음과 같이 설정하였다.

| 변수 | 표준값 $X_0$ | 공차 $\Delta X_{safe}$ | 근거 |
|---|---|---|---|
| 모재 표면온도 | 25 (기준 온도) | 10 C | 예열 없는 상온 용접에서 흔히 관찰되는 실내외 온도 편차 범위 |
| 유효 접촉각 | 시료별 설정각 | 5 도 | 전극 정렬 작업에서 통상 허용되는 기계적 공차 |
| 너겟 지름 | 목표 열입력에서의 예측 지름 | 0.30 mm | 목표 지름 대비 육안 및 계측 오차를 고려한 여유폭 |
| 인장강도 | 목표 열입력에서의 예측 강도 | 500 N | 실측 데이터의 시료 간 강도 편차 규모 |
| 전류 | baseline 전류의 중앙값 | 300 A | 장비 제어 정밀도를 고려한 통상적 전류 오차 범위 |
| 전압(유도) | baseline 전압의 중앙값 | 0.20 V | 접촉저항 변동에 따른 유도 전압의 자연스러운 흔들림 폭 |
| 통전시간 | baseline 통전시간의 중앙값 | 0.30 s | 제어기 응답 지연을 고려한 시간 오차 범위 |
| 가압력 | 80 psi | 15 psi | 전극 가압 실린더의 통상적 압력 제어 오차 범위 |
| 열입력 | 목표 열입력 $Q_{target}$ | $0.10\times Q_{target}$ | 목표값 대비 10% 이내를 공정 관리 목표 편차로 설정 |

위 공차값은 통계적으로 추정한 값이 아니라, 저항 점용접 공정에서 통상적으로 받아들여지는 관리
범위를 참고하여 설계자가 부여한 값이다. 즉 1행 3열과 2행 3열의 위험/경계/안전 구분은 **개별
변수가 표준값에서 얼마나 벗어났는지**를 기준으로 한 공차 기반 판정이다.

### 5.2 공정 위상도와 열입력 추이의 임계값 근거

2행 1열(공정 위상도)과 2행 2열(유효 열입력 추이)의 위험/경계/안전 구분은 위의 공차 기반 방식과
달리, 실측 493건에 로지스틱 회귀로 피팅한 미융착 확률 모형에서 직접 얻는다.

$$Q_{lo}=Q_{min}-\sigma_{Q_{min}},\qquad Q_{hi}=Q_{min}+\sigma_{Q_{min}}$$

* $Q_{min}$: 미융착 확률 $p_{bad}(Q)=1/(1+e^{k(Q-Q_{min})})$이 50%가 되는 열입력. 실측 데이터에
  곡선을 피팅하여 얻은 값이다.
* $\sigma_{Q_{min}}$: 곡선 피팅 과정에서 얻어지는 $Q_{min}$의 표준오차로, 유한한 표본으로부터
  추정한 값이 지니는 통계적 불확실성의 크기를 나타낸다.

| 상태 | 열입력 범위 | 색상 | 해석 |
|---|---|---|---|
| 위험 | $Q<Q_{lo}$ | 빨강 | 미융착 확률이 통계적으로 유의하게 50%를 상회하는 구간 |
| 경계 | $Q_{lo}\le Q<Q_{hi}$ | 노랑 | 피팅 불확실성 범위 내에 있어 미융착 여부를 단정할 수 없는 구간 |
| 안전 | $Q\ge Q_{hi}$ | 초록 | 미융착 확률이 통계적으로 유의하게 50% 미만인 구간 |

두 패널이 동일한 $Q_{lo}$, $Q_{hi}$를 공유하므로, 공정 위상도에서 안전 구역에 속한 동작점은
열입력 추이 그래프에서도 항상 초록 구간 위에 놓이게 되어 두 그래프의 해석이 서로 어긋나지
않는다.

### 5.3 패널 구성

| 위치 | 패널 | 표시 내용 |
|---|---|---|
| 1행 1열 | 실측 IR 열화상 | 각 프레임에 대응하는 실제 촬영 이미지와 시료 정보 |
| 1행 2열 | 비드 및 너겟 단면 | 예측 너겟 지름을 단면 형상으로 렌더링, baseline 대 optimal 비교 |
| 1행 3열 | 환경 및 품질 지수 | 모재 표면온도(용접 전, 예열/환경 조건), 유효 접촉각, 너겟 지름, 인장강도의 정규화 상태 |
| 2행 1열 | **공정 위상도** | 전류-통전시간 평면의 안전/경계/위험 구역, 실측 493건 분포, 실시간 동작점 |
| 2행 2열 | 유효 열입력 추이 | 미융착 위험/경계/안전 배경(2행1열과 동일 기준) 위에서 $Q_{eff}(t)$의 목표 추종 여부, 두 운전 방식 비교 |
| 2행 3열 | 장비 변수 지수 | 전류, 전압, 통전시간, 가압력, 열입력의 정규화 상태 |
| 3행 1열 | **피팅 함수와 동작점** | 미융착 확률 곡선과 너겟 성장 곡선, 실측 구간 통계, 현재 운전점 |
| 3행 2열 | 조작 변수 보정 궤적 | 제어기가 산출한 전류/통전시간/가압력의 시간 변화 |
| 3행 3열 | 결함 확률과 가드레일 | 종합 결함 확률의 시계열, 두 운전 방식의 성능 격차 |

### 5.4 전압의 산출

원본 데이터셋에는 전압이 계측되어 있지 않으므로, 옴의 법칙을 이용해 유도한다.

$$V_{eff}=I\cdot R_{contact}\cdot R_{eff}$$

* $V_{eff}$: 옴의 법칙으로 유도한 전압(V).
* $I$: 전류(A).
* $R_{contact}$: 접촉저항 규모를 나타내는 가정값(옴).
* $R_{eff}$: 유효 접촉저항의 상대값(3.1절 참고).

여기서 $R_{contact}$는 저항 점용접의 전형적인 접촉저항 규모를 반영한 가정값이다. 따라서 전압
패널은 **온도 외란이 전기적 관측량으로 드러나는 경로**를 보여주는 유도 지표이며, 실측값이 아니다.

### 5.5 종합 결함 확률 가드레일 기준

3행 3열의 위험/경계/안전 구간은 미융착과 폭발형 결함을 결합한 종합 결함 확률
$p_{total}=1-(1-p_{bad})(1-p_{explode})$의 백분율값에 대해 다음과 같은 고정 임계값을 사용한다.

| 상태 | 종합 결함 확률 | 색상 | 근거 |
|---|---|---|---|
| 안전 | 20% 미만 | 초록 | 결함 발생 가능성이 낮아 별도 조치가 필요하지 않은 구간 |
| 경계 | 20% 이상 50% 미만 | 노랑 | 결함 확률이 무시할 수 없는 수준으로 상승한 감시 강화 구간 |
| 위험 | 50% 이상 | 빨강 | 결함이 발생할 확률이 발생하지 않을 확률보다 높아지는 구간 |

50%는 확률의 정의상 결함 발생과 미발생의 우열이 뒤바뀌는 자연스러운 판정 경계이며, 20%는
모형과 외란 추정의 불확실성을 고려하여 50%에 도달하기 전에 미리 경계 신호를 주기 위해 설계자가
부여한 여유 기준선이다. 즉 1행 3열/2행 3열의 공차 기준, 2행 1열/2행 2열의 통계적 임계값 기준과
달리, 이 절의 기준은 안전공학에서 흔히 쓰이는 **가드레일형 설계 기준**에 해당한다.

In [ ]:
# ==========================================
# [Cell 5] 3x3 디지털 트윈 대시보드 렌더링
# ==========================================

# 접촉저항 규모 가정값 (저항 점용접의 전형적 크기, 실측값이 아님)
R_CONTACT_OHM = 5.0e-4

# 그래프 전역 스타일: 축 라벨/눈금 숫자/제목/범례를 굵고 크게 표시한다
plt.rcParams.update({
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.titlesize": 19,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})


class DashboardRenderer:
    """실측 IR 이미지와 폐루프 제어 로그를 3x3 애니메이션으로 렌더링한다."""

    def __init__(self, repo, frames, log, params, Q_target):
        self.repo = repo
        self.frames = frames
        self.log = log
        self.params = params
        self.Q_target = Q_target
        self.n = len(log)

        # 목표 품질 지표: 목표 열입력에서 얻어지는 너겟 지름과 인장강도
        self.D_target = float(nugget_growth_model(
            Q_target, params["D0"], params["Dmax"], params["tau"]))
        self.F_target = params["F_slope"] * self.D_target + params["F_intercept"]

        self._preload_images()
        self._prepare_indices()

    # ---------- 사전 준비 ----------
    def _preload_images(self):
        """모든 프레임의 IR 이미지를 미리 읽어 렌더링 지연을 줄인다."""
        self.images = [plt.imread(self.repo.image_path(sid))
                       for sid in self.log["sample_id"]]

    def _prepare_indices(self):
        """정규화 지수를 전 프레임에 대해 사전 계산한다.

        누적 최소/최대값을 함께 보관하여, 막대 위에 변동 범위를
        오차 막대 형태로 표시한다.
        """
        L = self.log
        # 유도 전압: 옴의 법칙으로 계산한 값이며 실측이 아니다
        V_base = L["base_current"] * R_CONTACT_OHM * L["R_eff_true"]
        V_opt = L["opt_current"] * R_CONTACT_OHM * L["R_eff_true"]
        self.V_base, self.V_opt = V_base.values, V_opt.values

        # (변수값, 표준값, 허용공차) 순서로 정의한다
        env_spec = [
            (T_REFERENCE_C + L["temp_dev_true"].values, T_REFERENCE_C, 10.0),
            (L["tilt_true"].values, L["angle_deg"].values, 5.0),
            (L["opt_nugget"].values, self.D_target, 0.30),
            (L["opt_tensile"].values, self.F_target, 500.0),
        ]
        mach_spec = [
            (L["opt_current"].values, float(np.median(L["base_current"])), 300.0),
            (self.V_opt, float(np.median(V_base)), 0.20),
            (L["opt_time"].values, float(np.median(L["base_time"])), 0.30),
            (L["opt_pressure"].values, 80.0, 15.0),
            (L["opt_Q"].values, self.Q_target, 0.10 * self.Q_target),
        ]
        self.env_index = np.array([100.0 + 10.0 * (x - x0) / tol
                                   for x, x0, tol in env_spec])
        self.mach_index = np.array([100.0 + 10.0 * (x - x0) / tol
                                    for x, x0, tol in mach_spec])
        self.env_labels = ["모재 표면온도\n(용접 전)", "유효\n접촉각", "너겟\n지름", "인장\n강도"]
        self.mach_labels = ["전류", "전압\n(유도)", "통전\n시간", "가압력", "열입력"]

        # 조작 변수 보정 비율 (3행 2열에서 사용): baseline 대비 optimal 값의 백분율.
        # 전체 프레임에 대해 미리 계산해 두면, 세로축 범위를 데이터가 잘리지
        # 않도록 실제 최소/최대값 기준으로 정할 수 있다.
        self.mv_ratio = {
            key: (L["opt_" + key].values / L["base_" + key].values * 100.0)
            for key in ("current", "time", "pressure")
        }

    @staticmethod
    def _state_color(index_value):
        """정규화 지수를 안전/경계/위험 색상으로 변환한다."""
        if index_value < CAUTION_LO or index_value > CAUTION_HI:
            return FG_RISK
        if index_value < SAFE_LO or index_value > SAFE_HI:
            return FG_CAUTION
        return FG_SAFE

    @staticmethod
    def _add_zone_bands(ax, lo=-1e4, hi=1e4):
        """정규화 지수 축에 안전/경계/위험 배경 띠를 그린다."""
        ax.axhspan(SAFE_LO, SAFE_HI, color=C_SAFE, zorder=0)
        ax.axhspan(CAUTION_LO, SAFE_LO, color=C_CAUTION, zorder=0)
        ax.axhspan(SAFE_HI, CAUTION_HI, color=C_CAUTION, zorder=0)
        ax.axhspan(lo, CAUTION_LO, color=C_RISK, zorder=0)
        ax.axhspan(CAUTION_HI, hi, color=C_RISK, zorder=0)
        ax.axhline(100.0, color="red", ls="--", lw=1.2, zorder=3)

    # ---------- 패널 구성 ----------
    def _setup(self):
        self.fig, axes = plt.subplots(3, 3, figsize=(22, 18), dpi=80)
        self.fig.suptitle(
            "실측 결합형 적응 용접 시뮬레이션: 실시간 최적 조건 탐색 대시보드",
            fontsize=24, fontweight="bold")
        plt.subplots_adjust(hspace=0.32, wspace=0.42, top=0.94, bottom=0.05)

        self.ax_img, self.ax_sec, self.ax_env = axes[0]
        self.ax_phase, self.ax_heat, self.ax_mach = axes[1]
        self.ax_fit, self.ax_mv, self.ax_def = axes[2]

        self._setup_image_panel()
        self._setup_section_panel()
        self._setup_env_panel()
        self._setup_phase_panel()
        self._setup_heat_panel()
        self._setup_machine_panel()
        self._setup_fit_panel()
        self._setup_mv_panel()
        self._setup_defect_panel()

    def _setup_image_panel(self):
        """1행 1열: 실측 IR 열화상 이미지."""
        ax = self.ax_img
        self.img_artist = ax.imshow(self.images[0])
        ax.axis("off")
        ax.set_title("1. 실측 IR 열화상 이미지", fontweight="bold")
        self.img_label = ax.text(
            0.03, 0.97, "", transform=ax.transAxes, color="white",
            fontsize=14, fontweight="bold", va="top",
            bbox=dict(boxstyle="round", facecolor="black", alpha=0.55))
        # 우측 상단: 전체 99프레임 중 몇 번째 이미지인지 나타내는 번호 라벨
        self.img_number_label = ax.text(
            0.97, 0.97, "", transform=ax.transAxes, color="white",
            fontsize=14, fontweight="bold", va="top", ha="right",
            bbox=dict(boxstyle="round", facecolor="black", alpha=0.55))
        # 이미지 우측 색상바: 사진 자체가 이미 색상 팔레트로 렌더링되어 있으므로,
        # 절대 온도가 아니라 상대적 열 강도를 나타내는 참고용 막대이다
        thermal_sm = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(0, 1))
        cbar = self.fig.colorbar(thermal_sm, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("상대 열 강도", fontweight="bold", fontsize=13)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(["낮음", "높음"])
        cbar.ax.tick_params(labelsize=12)

    def _setup_section_panel(self):
        """1행 2열: 비드 및 너겟 단면 형상."""
        ax = self.ax_sec
        ax.set_title("2. 비드 및 너겟 단면", fontweight="bold")
        ax.set_xlim(-6, 6); ax.set_ylim(-4, 4); ax.set_aspect("equal")
        ax.set_xlabel("폭 방향 (mm)"); ax.set_ylabel("깊이 (mm)")
        thickness = float(self.frames["thickness_avg_mm"].mean())
        ax.axhline(0, color="black", lw=2)
        ax.axhline(-thickness, color="black", lw=2, ls="--")
        # 두 운전 방식의 너겟을 겹쳐 그려 크기 차이를 직접 비교한다
        self.sec_base = Ellipse((0, -thickness / 2), 0, 0, facecolor="none",
                                edgecolor=C_BASELINE_NUGGET, lw=2.5, ls="--", zorder=3)
        self.sec_opt = Ellipse((0, -thickness / 2), 0, 0, facecolor=C_OPTIMAL,
                               alpha=0.55, edgecolor="black", lw=1.5, zorder=2)
        ax.add_patch(self.sec_base); ax.add_patch(self.sec_opt)
        ax.legend(handles=[
            mpatches.Patch(facecolor=C_OPTIMAL, alpha=0.55, label="optimal 너겟"),
            mpatches.Patch(facecolor="none", edgecolor=C_BASELINE_NUGGET, label="baseline 너겟"),
        ], loc="upper right")

    def _setup_env_panel(self):
        """1행 3열: 환경 및 품질 정규화 지수."""
        ax = self.ax_env
        ax.set_title("3. 환경 및 품질 지수 (표준 대비)", fontweight="bold")
        self.env_bars = ax.bar(self.env_labels, [100] * 4, width=0.6, zorder=2)
        self._add_zone_bands(ax)
        # 우측 상단: 배경색(위험/경계/안전)이 각각 무엇을 뜻하는지 알려주는 범례
        ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="위험"),
            mpatches.Patch(facecolor=C_CAUTION, label="경계"),
            mpatches.Patch(facecolor=C_SAFE, label="안전"),
        ], loc="upper right", framealpha=0.85)
        top = max(200.0, float(np.max(self.env_index)) * 1.2)
        ax.set_ylim(0, top)
        ax.set_ylabel("정규화 지수 (%)")
        self.env_texts = [ax.text(i, 100, "", ha="center", fontsize=13,
                                  fontweight="bold", zorder=4) for i in range(4)]
        self.env_bands = [ax.plot([], [], "k-", lw=1.5, zorder=4)[0]
                          for _ in range(4)]

    def _setup_phase_panel(self):
        """2행 1열: 공정 위상도 (전류 x 통전시간)."""
        ax = self.ax_phase
        ax.set_title("4. 공정 위상도 (전류 x 통전시간)", fontweight="bold")
        I_grid = np.linspace(500, 4500, 240)
        t_grid = np.linspace(0.05, 1.65, 240)
        II, TT = np.meshgrid(I_grid, t_grid)
        QQ = II ** 2 * TT     # 접촉저항을 1로 정규화한 열입력 지표
        q_lo = self.params["Q_min"] - self.params["Q_min_err"]
        q_hi = self.params["Q_min"] + self.params["Q_min_err"]
        # 열입력이 임계값 미만이면 미융착 위험 구역으로 분류한다
        zone = np.where(QQ < q_lo, 0, np.where(QQ < q_hi, 1, 2))
        ax.contourf(II, TT, zone, levels=[-0.5, 0.5, 1.5, 2.5],
                    colors=[C_RISK, C_CAUTION, C_SAFE], zorder=0)
        # 목표 열입력 등고선을 기준선으로 함께 표시한다
        cs = ax.contour(II, TT, QQ, levels=[self.Q_target], colors="black",
                        linestyles="--", linewidths=1.5, zorder=1)
        ax.clabel(cs, fmt={self.Q_target: "목표 Q"}, fontsize=12)
        # 실측 493건의 분포를 배경으로 표시한다
        ax.scatter(self.repo.samples["avg_current_A"],
                   self.repo.samples["weld_time_s"],
                   s=10, c="#555555", alpha=0.25, zorder=2, label="실측 493건")
        self.phase_trail, = ax.plot([], [], color=C_OPTIMAL, lw=1.0,
                                    alpha=0.5, zorder=3)
        self.phase_base, = ax.plot([], [], "o", ms=12, color=C_BASELINE,
                                   mec="black", zorder=5, label="baseline")
        self.phase_opt, = ax.plot([], [], "o", ms=12, color=C_OPTIMAL,
                                  mec="black", zorder=6, label="optimal")
        ax.set_xlim(500, 4500); ax.set_ylim(0.05, 1.65)
        ax.set_xlabel("전류 (A)"); ax.set_ylabel("통전시간 (s)")
        # 좌측 가운데: 배경색(위험/경계/안전)이 각각 무엇을 뜻하는지 알려주는 범례.
        # add_artist로 고정해 두어야 아래 두 번째 범례를 그릴 때 지워지지 않는다
        zone_legend_phase = ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="위험"),
            mpatches.Patch(facecolor=C_CAUTION, label="경계"),
            mpatches.Patch(facecolor=C_SAFE, label="안전"),
        ],
                                      loc="center left", framealpha=0.85)
        ax.add_artist(zone_legend_phase)
        ax.legend(loc="upper right", framealpha=0.85)
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_heat_panel(self):
        """2행 2열: 유효 열입력 추이.

        배경 위험/경계/안전 구간은 2행 1열 공정 위상도와 동일한 미융착 임계값
        (Q_min +/- 오차)을 기준으로 한다. 같은 열입력 Q를 다루는 두 패널이 같은
        기준으로 색칠되어야 두 그래프를 함께 볼 때 해석이 일관된다.
        """
        ax = self.ax_heat
        ax.set_title("5. 유효 열입력 $Q_{eff}$ 추이", fontweight="bold")
        ax.set_xlim(0, self.n - 1)
        y_top = max(self.log["base_Q"].max(), self.log["opt_Q"].max()) * 1.1
        ax.set_ylim(0, y_top)
        # 미융착 위험/경계/안전 구간을 배경으로 표시한다 (2행1열과 동일 기준)
        q_lo = self.params["Q_min"] - self.params["Q_min_err"]
        q_hi = self.params["Q_min"] + self.params["Q_min_err"]
        ax.axhspan(0, q_lo, color=C_RISK, zorder=0)
        ax.axhspan(q_lo, q_hi, color=C_CAUTION, zorder=0)
        ax.axhspan(q_hi, y_top, color=C_SAFE, zorder=0)
        ax.axhline(self.Q_target, color="black", ls="--", lw=1.5, zorder=2,
                   label="목표 $Q$")
        self.heat_base, = ax.plot([], [], color=C_BASELINE, lw=1.5,
                                  label="baseline")
        self.heat_opt, = ax.plot([], [], color=C_OPTIMAL, lw=2.0,
                                 label="optimal")
        ax.set_xlabel("프레임 (시료 순번)"); ax.set_ylabel("열입력 지표")
        # 우측 가운데: 배경색(위험/경계/안전)이 각각 무엇을 뜻하는지 알려주는 범례
        zone_legend_heat = ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="위험"),
            mpatches.Patch(facecolor=C_CAUTION, label="경계"),
            mpatches.Patch(facecolor=C_SAFE, label="안전"),
        ], loc="center right", framealpha=0.85)
        ax.add_artist(zone_legend_heat)
        ax.legend(loc="lower right")
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_machine_panel(self):
        """2행 3열: 장비 제어 변수 정규화 지수."""
        ax = self.ax_mach
        ax.set_title("6. 장비 제어 변수 지수 (표준 대비)", fontweight="bold")
        self.mach_bars = ax.bar(self.mach_labels, [100] * 5, width=0.6, zorder=2)
        self._add_zone_bands(ax)
        # 좌측 상단: 배경색(위험/경계/안전)이 각각 무엇을 뜻하는지 알려주는 범례
        ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="위험"),
            mpatches.Patch(facecolor=C_CAUTION, label="경계"),
            mpatches.Patch(facecolor=C_SAFE, label="안전"),
        ],
                  loc="upper right", framealpha=0.85)
        top = max(200.0, float(np.max(self.mach_index)) * 1.2)
        ax.set_ylim(0, top)
        ax.set_ylabel("정규화 지수 (%)")
        ax.tick_params(axis="x", labelsize=15)
        self.mach_texts = [ax.text(i, 100, "", ha="center", fontsize=13,
                                   fontweight="bold", zorder=4) for i in range(5)]
        self.mach_bands = [ax.plot([], [], "k-", lw=1.5, zorder=4)[0]
                           for _ in range(5)]

    def _setup_fit_panel(self):
        """3행 1열: 피팅 함수와 실시간 동작점."""
        ax = self.ax_fit
        ax.set_title("7. 피팅 함수 위의 실시간 동작점", fontweight="bold")
        q_max = max(self.log["base_Q"].max(), self.log["opt_Q"].max()) * 1.15
        q_axis = np.linspace(1e4, q_max, 400)
        # 왼쪽 축: 미융착 확률 곡선과 실측 구간 통계
        ax.plot(q_axis, bad_probability_model(
            q_axis, self.params["Q_min"], self.params["k_bad"]),
            color=FG_SAFE, lw=2, label="미융착 확률 $p_{bad}(Q)$")
        b = self.repo.binned
        ax.errorbar(b["heat_mean"], b["bad_rate"], yerr=b["bad_rate_sem"],
                    fmt="o", ms=5, color=FG_SAFE, ecolor="#d62728",
                    capsize=3, alpha=0.8, label="실측 구간 평균 +/- SEM")
        ax.set_xlim(0, q_max); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("열입력 지표 $Q$"); ax.set_ylabel("미융착 확률")
        # 오른쪽 축: 너겟 성장 곡선
        ax2 = ax.twinx()
        ax2.plot(q_axis, nugget_growth_model(
            q_axis, self.params["D0"], self.params["Dmax"], self.params["tau"]),
            color="#9467bd", lw=2, ls="-.", label="너겟 지름 $D(Q)$")
        # 우측 축 라벨과 눈금 숫자를 너겟 지름 곡선과 같은 보라색으로 맞춘다
        ax2.set_ylabel("너겟 지름 (mm)", color="#9467bd")
        ax2.tick_params(axis="y", labelcolor="#9467bd")
        ax2.set_ylim(2.5, self.params["Dmax"] * 1.15)
        self.fit_base, = ax.plot([], [], "o", ms=12, color=C_BASELINE,
                                 mec="black", zorder=5)
        self.fit_opt, = ax.plot([], [], "o", ms=12, color=C_OPTIMAL,
                                mec="black", zorder=6)
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="upper right")
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_mv_panel(self):
        """3행 2열: 조작 변수 보정 궤적.

        통전시간은 변동 폭이 작고(초 단위), 전류와 가압력은 변동 폭이 상대적으로
        크므로, 좌측 축에 통전시간을, 우측 축에 전류와 가압력을 배치하여 세 계열이
        서로의 시인성을 해치지 않도록 한다.
        """
        ax = self.ax_mv
        ax.set_title("8. 조작 변수 실시간 보정 궤적", fontweight="bold")
        ax.set_xlim(0, self.n - 1)

        # 좌측 축: 통전시간 보정 비율
        t_ratio = self.mv_ratio["time"]
        t_lo = min(0.0, float(t_ratio.min()) * 0.9)
        t_hi = float(t_ratio.max()) * 1.15
        ax.set_ylim(t_lo, t_hi)
        ax.axhline(100, color="#2ca02c", ls="--", lw=1.2, zorder=1)
        ax.set_xlabel("프레임 (시료 순번)")
        ax.set_ylabel("통전시간 비율 (%)", color="#2ca02c")
        ax.tick_params(axis="y", labelcolor="#2ca02c")
        self.mv_lines = {}
        self.mv_lines["time"], = ax.plot([], [], color="#2ca02c", lw=1.8,
                                         label="통전시간(좌축)")

        # 우측 축: 전류 및 가압력 보정 비율
        ax2 = ax.twinx()
        cp_ratio = np.concatenate([self.mv_ratio["current"], self.mv_ratio["pressure"]])
        cp_lo = min(0.0, float(cp_ratio.min()) * 0.9)
        cp_hi = float(cp_ratio.max()) * 1.15
        ax2.set_ylim(cp_lo, cp_hi)
        ax2.axhline(100, color="black", ls=":", lw=1.2, zorder=1)
        ax2.set_ylabel("전류 · 가압력 비율 (%)")
        for key, color, label in [("current", "#d62728", "전류(우축)"),
                                  ("pressure", "#ff7f0e", "가압력(우축)")]:
            self.mv_lines[key], = ax2.plot([], [], color=color, lw=1.8, label=label)
        self.ax_mv2 = ax2

        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="upper right", ncol=1)
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_defect_panel(self):
        """3행 3열: 종합 결함 확률과 가드레일 성능."""
        ax = self.ax_def
        ax.set_title("9. 종합 결함 확률과 가드레일", fontweight="bold")
        ax.set_xlim(0, self.n - 1); ax.set_ylim(0, 105)
        ax.axhspan(0, 20, color=C_SAFE, zorder=0)
        ax.axhspan(20, 50, color=C_CAUTION, zorder=0)
        ax.axhspan(50, 105, color=C_RISK, zorder=0)
        self.def_base, = ax.plot([], [], color=C_BASELINE, lw=1.5,
                                 label="baseline")
        self.def_opt, = ax.plot([], [], color=C_OPTIMAL, lw=2.0,
                                label="optimal")
        self.def_text = ax.text(0.98, 0.95, "", transform=ax.transAxes,
                                ha="right", va="top", fontsize=18,
                                fontweight="bold",
                                bbox=dict(boxstyle="round", facecolor="white",
                                          alpha=0.85))
        ax.set_xlabel("프레임 (시료 순번)"); ax.set_ylabel("결함 확률 (%)")
        # 우측 가운데: 배경색(위험/경계/안전)이 각각 무엇을 뜻하는지 알려주는 범례.
        # add_artist로 고정해 두어야 아래 baseline/optimal 범례를 그릴 때 지워지지 않는다
        zone_legend_def = ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="위험"),
            mpatches.Patch(facecolor=C_CAUTION, label="경계"),
            mpatches.Patch(facecolor=C_SAFE, label="안전"),
        ],
                                    loc="center right", framealpha=0.85)
        ax.add_artist(zone_legend_def)
        # 기본 좌측 상단 위치(약 x=0, y=105)에서 가로 +10, 세로 -3 만큼
        # 데이터 좌표 기준으로 이동한다
        ax.legend(loc="upper left", bbox_to_anchor=(10, 102),
                  bbox_transform=ax.transData)
        ax.grid(True, ls=":", alpha=0.4)

    # ---------- 프레임 갱신 ----------
    def _update(self, i):
        row = self.log.iloc[i]
        past = slice(0, i + 1)
        x_axis = np.arange(i + 1)

        # 1. IR 이미지
        self.img_artist.set_data(self.images[i])
        self.img_label.set_text(
            "시료 #%d  |  %s\n전극각도 %.0f도"
            % (row["sample_id"], row["category"], row["angle_deg"]))
        self.img_number_label.set_text("이미지 %03d / %03d" % (i + 1, self.n))

        # 2. 너겟 단면 (지름을 폭, 그 60%를 높이로 근사한다)
        for patch, d in [(self.sec_base, row["base_nugget"]),
                         (self.sec_opt, row["opt_nugget"])]:
            patch.width, patch.height = d, d * 0.6

        # 3. 환경 및 품질 지수
        env_now = self.env_index[:, i]
        env_raw = [T_REFERENCE_C + row["temp_dev_true"], row["tilt_true"],
                   row["opt_nugget"], row["opt_tensile"]]
        env_fmt = ["%.1f도", "%.1f도", "%.2fmm", "%.0fN"]
        for k, bar in enumerate(self.env_bars):
            bar.set_height(env_now[k])
            bar.set_color(self._state_color(env_now[k]))
            self.env_texts[k].set_text(env_fmt[k] % env_raw[k])
            self.env_texts[k].set_position((k, env_now[k] * 1.02))
            # 지금까지 관측된 변동 범위를 세로선으로 표시한다
            self.env_bands[k].set_data(
                [k, k], [self.env_index[k, past].min(),
                         self.env_index[k, past].max()])

        # 4. 공정 위상도
        self.phase_base.set_data([row["base_current"]], [row["base_time"]])
        self.phase_opt.set_data([row["opt_current"]], [row["opt_time"]])
        self.phase_trail.set_data(self.log["opt_current"].values[past],
                                  self.log["opt_time"].values[past])

        # 5. 유효 열입력
        self.heat_base.set_data(x_axis, self.log["base_Q"].values[past])
        self.heat_opt.set_data(x_axis, self.log["opt_Q"].values[past])

        # 6. 장비 제어 변수 지수
        mach_now = self.mach_index[:, i]
        mach_raw = [row["opt_current"], self.V_opt[i], row["opt_time"],
                    row["opt_pressure"], row["opt_Q"]]
        mach_fmt = ["%.0fA", "%.2fV", "%.2fs", "%.0fpsi", "%.1e"]
        for k, bar in enumerate(self.mach_bars):
            bar.set_height(mach_now[k])
            bar.set_color(self._state_color(mach_now[k]))
            self.mach_texts[k].set_text(mach_fmt[k] % mach_raw[k])
            self.mach_texts[k].set_position((k, mach_now[k] * 1.02))
            self.mach_bands[k].set_data(
                [k, k], [self.mach_index[k, past].min(),
                         self.mach_index[k, past].max()])

        # 7. 피팅 함수 위의 동작점
        self.fit_base.set_data([row["base_Q"]], [row["base_p_bad"]])
        self.fit_opt.set_data([row["opt_Q"]], [row["opt_p_bad"]])

        # 8. 조작 변수 보정 비율 (baseline 설정값을 100%로 둔다, 사전 계산값 사용)
        for key in ("current", "time", "pressure"):
            self.mv_lines[key].set_data(x_axis, self.mv_ratio[key][past])

        # 9. 종합 결함 확률
        self.def_base.set_data(x_axis, self.log["base_p_total"].values[past] * 100)
        self.def_opt.set_data(x_axis, self.log["opt_p_total"].values[past] * 100)
        self.def_text.set_text(
            "baseline %.1f%%\noptimal  %.1f%%"
            % (row["base_p_total"] * 100, row["opt_p_total"] * 100))

    def render(self, gif_path, fps=8):
        self._setup()
        anim = animation.FuncAnimation(self.fig, self._update,
                                       frames=self.n, interval=1000 // fps,
                                       blit=False)
        anim.save(gif_path, writer="pillow", fps=fps)
        plt.close(self.fig)
        return anim


# --- 실행 ---------------------------------------------------------
plt.rcParams["animation.embed_limit"] = 200

renderer = DashboardRenderer(repo, frames, log, params, Q_TARGET)
gif_path = os.path.join(FIGURE_DIR, "3x3_weld_dashboard.gif")
anim = renderer.render(gif_path)

print("[완료] %d 프레임 대시보드 저장: %s" % (len(log), gif_path))
display(HTML(anim.to_jshtml()))

## 6. 결과 분석: 9개 패널의 해석

본 절은 `result_sim_weld/figures/3x3_weld_dashboard.gif`의 각 패널이 무엇을 보여주는지,
그리고 그 결과가 제어 구조의 어떤 성질에서 비롯되는지를 항목별로 정리한다.

> **수치 확인 안내**: 아래 기술된 정량값은 노트북 작성 시점에 동일한 알고리즘을 독립적으로
> 재현하여 얻은 **예비 검증 결과**이다. 난수 생성기와 피팅 알고리즘의 세부 구현 차이로 인해
> 실제 실행 결과와 소수점 단위 차이가 있을 수 있으므로, 노트북 실행 후 `step5_summary.csv`의
> 값으로 대조하여야 한다.

---

### 6.1 실시간 공정 관측 (1행)

**패널 1. 실측 IR 열화상 이미지**
*한 줄 요약*: 매 프레임 실제로 촬영된 열화상 이미지가 교체되며, 그 프레임이 어떤 실측 시료에
근거하는지를 명시한다.

99개 프레임은 IR 이미지를 보유한 실측 시료 99건과 1대 1로 대응하며, 열입력 오름차순으로
정렬되어 있다. 따라서 재생이 진행될수록 화면의 용융 흔적이 점차 뚜렷해지는 경향을 관찰할 수
있다. 좌측 상단에는 시료 번호, 결함 분류, 설계 전극각도가 함께 표시되어 이후 패널의 수치가
어떤 조건에서 산출된 것인지 추적할 수 있다. 이 이미지는 적외선 카메라로 촬영된 뒤 이미
색상 팔레트로 렌더링된 사진이며, 절대 온도 계측값이 아니라 상대적 열 분포를 나타낸다.

**패널 2. 비드 및 너겟 단면**
*한 줄 요약*: 두 운전 방식이 만들어내는 너겟 크기를 같은 좌표계에 겹쳐 그려, 제어의 효과를
형상 차원에서 직접 비교한다.

파란 채움은 최적 제어의 예측 너겟, 회색 점선은 무보정 운전의 예측 너겟이다. 두 지름 모두
너겟 성장 곡선 $D(Q)$로 산출되므로, 열입력이 부족한 프레임에서는 회색 점선이 뚜렷하게
작아지는 반면 파란 영역은 목표 크기를 유지한다. 곡선이 $D_{max}$로 포화하는 성질 때문에
열입력이 충분한 구간에서는 두 형상이 거의 겹치며, 이는 제어의 이득이 **저열입력 구간에
집중**됨을 시각적으로 보여준다.

**패널 3. 환경 및 품질 정규화 지수**
*한 줄 요약*: 제어가 불가능한 환경 변수는 시간이 갈수록 공차를 이탈하는 반면, 제어의 결과인
품질 변수는 표준 범위 안에 유지된다.

네 개 막대 중 표면온도와 유효 접촉각은 **외란 그 자체**이므로 제어기가 직접 되돌릴 수 없고,
누적 랜덤워크의 성질에 따라 공정 후반으로 갈수록 100%에서 멀어진다. 반면 너겟 지름과
인장강도는 제어의 **결과**이므로 안전 범위 안에 머무른다. 각 막대에 붙은 세로선은 해당
시점까지 관측된 지수의 최소-최대 범위로, 변동 폭이 얼마나 누적되었는지를 나타낸다.

---

### 6.2 공정 상태 공간과 에너지 추적 (2행)

**패널 4. 공정 위상도**
*한 줄 요약*: 전류와 통전시간이 만드는 2차원 평면 위에서, 제어기가 동작점을 위험 구역 밖으로
이동시키는 궤적을 실측 분포와 함께 보여준다.

배경의 적색 구역은 열입력이 미융착 임계값 $Q_{min}$ 미만인 영역이며, 그 경계는 실측 493건에서
피팅한 값이다. 회색 점은 실측 493건의 실제 운전점 분포로, 이 시뮬레이션이 현실적인 운전 범위
안에서 이루어지고 있음을 뒷받침한다. 회색 표식(baseline)이 좌측 하단의 위험 구역에 머무르는
프레임에서, 파란 표식(optimal)은 전류와 통전시간을 함께 높여 안전 구역으로 이동한다.
파란 실선은 최적 동작점이 지나온 궤적이다.

**패널 5. 유효 열입력 추이**
*한 줄 요약*: 무보정 운전의 열입력은 시료마다 크게 흩어지지만, 최적 제어의 열입력은 목표 밴드에
밀착하여 유지된다.

프레임이 열입력 오름차순으로 정렬되어 있으므로 회색 곡선(baseline)은 우상향하는 계단 형태를
보인다. 반면 파란 곡선(optimal)은 목표 열입력 근방의 초록 밴드 안에 수렴한다. 이 패널은
제어기가 **온도 외란에 의한 접촉저항 변화를 전류와 통전시간으로 상쇄**하고 있음을 가장 직접적
으로 보여주는 증거이다.

**패널 6. 장비 제어 변수 지수**
*한 줄 요약*: 제어기가 산출한 조작 변수들이 공정 표준 대비 어느 위치에 있는지를 한눈에 제시한다.

전류, 유도 전압, 통전시간, 가압력, 열입력의 다섯 지표를 동일한 정규화 척도로 표시한다. 이 중
전압은 옴의 법칙으로 유도한 값이므로 온도 외란이 전기적 관측량으로 전이되는 경로를 나타낸다.
가압력 막대는 학습으로 찾은 최적값 부근에 고정되는 경향을 보이는데, 그 이유는 6.3절의
패널 8에서 설명한다.

---

### 6.3 제어 응답과 최종 성능 (3행)

**패널 7. 피팅 함수 위의 실시간 동작점**
*한 줄 요약*: 실측으로 학습한 두 곡선 위에 현재 운전점을 표시하여, 제어가 곡선의 어느 구간으로
동작점을 옮기고 있는지 보여준다.

초록 곡선은 미융착 확률 $p_{bad}(Q)$, 보라 점선은 너겟 성장 곡선 $D(Q)$이며, 초록 오차 막대는
실측 구간별 평균과 표준오차이다. 오차 막대가 곡선을 따라 배치되어 있다는 사실은 이 곡선이
임의로 그은 선이 아니라 **실제 관측 데이터에 피팅된 결과**임을 뒷받침한다. 회색 표식이 곡선의
급경사 구간(고위험)에 놓인 프레임에서, 파란 표식은 확률이 0에 가까운 평탄 구간으로 이동한다.

**패널 8. 조작 변수 실시간 보정 궤적**
*한 줄 요약*: 전류와 통전시간은 프레임마다 능동적으로 변하는 반면, 가압력은 학습된 최적값으로
이동한 뒤 일정하게 유지된다.

이 대비는 본 제어 구조의 성질을 그대로 드러낸다. 열입력 축(전류, 통전시간)은 매 프레임 달라지는
온도 외란과 시료 조건에 반응해야 하므로 지속적으로 조정된다. 반면 가압력 축은 팽출 확률 모델의
로짓이 접촉각 항과 가압력 항의 **덧셈 구조**로 되어 있어, 접촉각이 얼마이든 최적 가압력이
동일하게 산출된다. 예비 검증에서 최적 가압력은 전 프레임에 걸쳐 약 72.5 psi로 수렴하였다.

이는 알고리즘의 결함이 아니라 **데이터의 제약에서 비롯된 모델 구조의 귀결**이다. 접촉각과
가압력의 교호작용항을 도입하면 접촉각에 따라 최적 가압력이 달라지겠으나, 실측 데이터에는
전극각도 15도 조건에서 80 psi로 시공된 표본이 존재하지 않아 교호작용을 신뢰성 있게 추정할 수
없다. 따라서 본 연구는 데이터가 지지하는 범위에서 덧셈 모델을 채택하였다.

**패널 9. 종합 결함 확률과 가드레일**
*한 줄 요약*: 최적 제어는 무보정 운전 대비 종합 결함 확률을 약 70% 낮추며, 선행 연구에서
나타났던 확률의 대진폭 진동이 관측되지 않는다.

예비 검증 결과는 다음과 같다.

| 지표 | baseline | optimal | 감소율 |
|---|---|---|---|
| 미융착 확률 평균 | 0.051 | 0.003 | 94% |
| 팽출 확률 평균 | 0.025 | 0.019 | 23% |
| **종합 결함 확률 평균** | **0.071** | **0.022** | **69.5%** |

미융착 축의 개선폭이 압도적으로 큰 이유는, 전류와 통전시간이라는 두 조작 변수가 열입력을
직접적이고 강하게 지배하기 때문이다. 반면 팽출 축의 개선이 23%에 그친 것은, 가압력이 접촉각
증가를 부분적으로만 상쇄할 수 있고 접촉각 자체를 교정하는 구동기는 여전히 부재하기 때문이다.

## 7. 요약 및 결론

### 7.1 연구 요약

본 연구는 저항 점용접 실측 493건에서 학습한 물리 모델을 기반으로, 공정 중 발생하는 확률적
외란에 대해 **조작 변수를 실시간으로 재탐색하여 대응**하는 폐루프 제어 시스템을 구현하고,
이를 3x3 디지털 트윈 대시보드로 시각화하였다.

핵심 설계는 두 결함 유형이 서로 다른 물리 인자에 지배된다는 실측 관찰에서 출발한다.

$$\text{미융착} \leftarrow \text{열입력 } Q=I^2R_{eff}t \leftarrow (\text{전류},\ \text{통전시간})$$
$$\text{팽출} \leftarrow (\text{접촉각},\ \text{가압력}) \leftarrow \text{가압력}$$

이에 따라 전류와 통전시간으로 열입력 축을, 가압력으로 팽출 축을 담당하게 하는
**다변수 제어 구조**를 채택하였다.

### 7.2 주요 결과

1. **가압력의 최적점이 실측 데이터에 존재함을 확인하였다.** 가압력에 대한 결함률은 U자형
   관계를 보이며(35 psi에서 18.8%, 80 psi에서 1.5%, 95 psi에서 11.8%), 2차항을 포함한
   로지스틱 모델이 이 관계를 재현하여 약 70 psi 부근의 내부 최적점을 산출하였다.

2. **종합 결함 확률을 약 70% 감소시켰다.** 무보정 운전 대비 미융착 확률은 94%, 팽출 확률은
   23% 감소하였으며, 종합 결함 확률은 0.071에서 0.022로 낮아졌다.

3. **선행 연구의 진동 문제가 해소되었다.** `4_sim.ipynb`에서는 조작 변수가 단일 축(토치 속도)
   이었기 때문에 팽출 위험이 제어되지 않아 결함 확률이 0에서 100% 사이를 반복적으로 오갔다.
   본 연구는 팽출 축에 독립적인 조작 변수를 부여함으로써 이 문제를 구조적으로 제거하였다.

### 7.3 한계

1. **표면온도와 굴곡 편차는 합성값이다.** 원본 데이터셋에 해당 항목이 계측되어 있지 않아
   물리적으로 타당한 범위에서 가정한 확률 과정으로 생성하였다. 따라서 개선폭의 절대값은
   주입한 외란의 크기에 좌우되며, 실제 현장 성능을 보증하지 않는다.

2. **최적 가압력이 접촉각에 반응하지 않는다.** 팽출 모델의 로짓이 덧셈 구조이므로 최적 가압력이
   상수로 수렴한다. 교호작용항을 도입하려면 전극각도 15도, 가압력 80 psi 조건의 실측 표본이
   필요하나 현 데이터셋에는 존재하지 않는다.

3. **접촉각 자체를 교정하는 구동기가 없다.** 굴곡으로 인한 접촉각 편차는 감지되지만 물리적으로
   되돌릴 수단이 없어, 가압력을 통한 간접적 위험 완화만 가능하다. 이것이 팽출 축 개선폭이
   미융착 축에 비해 작은 근본 원인이다.

4. **저항 온도계수는 문헌 통용값이다.** $\alpha=0.004\ [1/^\circ\mathrm{C}]$는 철강 계열의
   전형적 근사값이며 이 데이터셋에서 측정된 값이 아니다.

### 7.4 향후 과제

1. 전극 자세를 능동 보정하는 구동기를 도입하여 접촉각 편차를 직접 교정하고, 팽출 축의
   개선폭을 미융착 축 수준으로 끌어올린다.
2. 전극각도와 가압력의 교호작용을 신뢰성 있게 추정할 수 있도록, 현 데이터셋에 결측된
   조건(15도 x 80 psi 등)의 실측 표본을 보강한다.
3. 격자 탐색을 연속 최적화 기법으로 대체하여 탐색 해상도의 제약을 제거하고, 조작 변수의
   급격한 변화를 억제하는 변화율 제약을 목적함수에 추가한다.
4. 실제 온도 센서와 비전 기반 곡률 측정 장비를 확보하여 합성 외란을 실측으로 교체하고,
   본 제어 구조를 실제 데이터로 재검증한다.

### 7.5 산출물

| 파일 | 내용 |
|---|---|
| `step1_fitted_models.csv` | 네 가지 물리 모델의 피팅 파라미터 |
| `step2_explode_pressure_stats.csv` | 전극각도-가압력 교차 집계 (팽출 모델의 근거) |
| `step3_disturbance.csv` | 합성 외란 및 센서 관측 시계열 |
| `step4_control_log.csv` | 프레임별 두 운전 방식의 전체 기록 |
| `step5_summary.csv` | 최종 성능 비교 요약 |
| `figures/3x3_weld_dashboard.gif` | 9패널 디지털 트윈 애니메이션 (국문 표기) |
| `figures/3x3_weld_dashboard_Eng.gif` | 9패널 디지털 트윈 애니메이션 (영문 표기, 8절) |

## 8. 영어 표기 버전 대시보드

위 [Cell 5]와 동일한 `DashboardRenderer` 구조를 각 패널의 제목, 축 라벨, 범례 텍스트만
영어로 표기하여 재구성한 셀이다. 두 버전은 동일한 데이터와 동일한 산출 로직을 사용하며,
차이는 표시 언어뿐이다. 국문 독자를 위한 `3x3_weld_dashboard.gif`와 영문 독자를 위한
`3x3_weld_dashboard_Eng.gif`를 모두 보존하기 위하여 별도 셀로 분리하였다.

In [ ]:
# ==========================================
# [Cell 5-Eng] 3x3 Digital Twin Dashboard Rendering
# ==========================================

# Assumed contact resistance (typical size for resistance spot welding, not measured)
R_CONTACT_OHM = 5.0e-4

# Global graph style: bold and large axis labels/ticks/titles/legends
plt.rcParams.update({
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.titlesize": 19,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})


class DashboardRenderer:
    """Renders real IR images and closed-loop control logs into a 3x3 animation."""

    def __init__(self, repo, frames, log, params, Q_target):
        self.repo = repo
        self.frames = frames
        self.log = log
        self.params = params
        self.Q_target = Q_target
        self.n = len(log)

        # Target quality metrics: nugget diameter and tensile strength obtained from target heat input
        self.D_target = float(nugget_growth_model(
            Q_target, params["D0"], params["Dmax"], params["tau"]))
        self.F_target = params["F_slope"] * self.D_target + params["F_intercept"]

        self._preload_images()
        self._prepare_indices()

    # ---------- Preparation ----------
    def _preload_images(self):
        """Preloads IR images for all frames to reduce rendering latency."""
        self.images = [plt.imread(self.repo.image_path(sid))
                       for sid in self.log["sample_id"]]

    def _prepare_indices(self):
        """Pre-calculates normalization indices for all frames.

        Keeps cumulative min/max values to display the variation range
        as error bars on the bars.
        """
        L = self.log
        # Induced voltage: calculated by Ohm's law, not measured
        V_base = L["base_current"] * R_CONTACT_OHM * L["R_eff_true"]
        V_opt = L["opt_current"] * R_CONTACT_OHM * L["R_eff_true"]
        self.V_base, self.V_opt = V_base.values, V_opt.values

        # Defined in order: (variable value, standard value, tolerance)
        env_spec = [
            (T_REFERENCE_C + L["temp_dev_true"].values, T_REFERENCE_C, 10.0),
            (L["tilt_true"].values, L["angle_deg"].values, 5.0),
            (L["opt_nugget"].values, self.D_target, 0.30),
            (L["opt_tensile"].values, self.F_target, 500.0),
        ]
        mach_spec = [
            (L["opt_current"].values, float(np.median(L["base_current"])), 300.0),
            (self.V_opt, float(np.median(V_base)), 0.20),
            (L["opt_time"].values, float(np.median(L["base_time"])), 0.30),
            (L["opt_pressure"].values, 80.0, 15.0),
            (L["opt_Q"].values, self.Q_target, 0.10 * self.Q_target),
        ]
        self.env_index = np.array([100.0 + 10.0 * (x - x0) / tol
                                   for x, x0, tol in env_spec])
        self.mach_index = np.array([100.0 + 10.0 * (x - x0) / tol
                                    for x, x0, tol in mach_spec])
        self.env_labels = ["Surface Temp\n(Pre-weld)", "Effective\nAngle", "Nugget\nDiameter", "Tensile\nStrength"]
        self.mach_labels = ["Current", "Voltage\n(Induced)", "Weld\nTime", "Pressure", "Heat Input"]

        # Manipulated variable correction ratio (used in row 3, col 2): percentage of optimal vs baseline.
        # Pre-calculating for all frames allows setting the vertical axis range
        # based on actual min/max values to prevent clipping.
        self.mv_ratio = {
            key: (L["opt_" + key].values / L["base_" + key].values * 100.0)
            for key in ("current", "time", "pressure")
        }

    @staticmethod
    def _state_color(index_value):
        """Converts normalization index to safe/caution/risk colors."""
        if index_value < CAUTION_LO or index_value > CAUTION_HI:
            return FG_RISK
        if index_value < SAFE_LO or index_value > SAFE_HI:
            return FG_CAUTION
        return FG_SAFE

    @staticmethod
    def _add_zone_bands(ax, lo=-1e4, hi=1e4):
        """Draws safe/caution/risk background bands on the normalization index axis."""
        ax.axhspan(SAFE_LO, SAFE_HI, color=C_SAFE, zorder=0)
        ax.axhspan(CAUTION_LO, SAFE_LO, color=C_CAUTION, zorder=0)
        ax.axhspan(SAFE_HI, CAUTION_HI, color=C_CAUTION, zorder=0)
        ax.axhspan(lo, CAUTION_LO, color=C_RISK, zorder=0)
        ax.axhspan(CAUTION_HI, hi, color=C_RISK, zorder=0)
        ax.axhline(100.0, color="red", ls="--", lw=1.2, zorder=3)

    # ---------- Panel Setup ----------
    def _setup(self):
        self.fig, axes = plt.subplots(3, 3, figsize=(22, 18), dpi=80)
        self.fig.suptitle(
            "Empirical Adaptive Welding Simulation: Real-time Optimal Condition Dashboard",
            fontsize=24, fontweight="bold")
        plt.subplots_adjust(hspace=0.32, wspace=0.42, top=0.94, bottom=0.05)

        self.ax_img, self.ax_sec, self.ax_env = axes[0]
        self.ax_phase, self.ax_heat, self.ax_mach = axes[1]
        self.ax_fit, self.ax_mv, self.ax_def = axes[2]

        self._setup_image_panel()
        self._setup_section_panel()
        self._setup_env_panel()
        self._setup_phase_panel()
        self._setup_heat_panel()
        self._setup_machine_panel()
        self._setup_fit_panel()
        self._setup_mv_panel()
        self._setup_defect_panel()

    def _setup_image_panel(self):
        """Row 1, Col 1: Empirical IR Thermal Image."""
        ax = self.ax_img
        self.img_artist = ax.imshow(self.images[0])
        ax.axis("off")
        ax.set_title("1. Empirical IR Thermal Image", fontweight="bold")
        self.img_label = ax.text(
            0.03, 0.97, "", transform=ax.transAxes, color="white",
            fontsize=14, fontweight="bold", va="top",
            bbox=dict(boxstyle="round", facecolor="black", alpha=0.55))
        # Top right: number label showing the current frame out of total frames
        self.img_number_label = ax.text(
            0.97, 0.97, "", transform=ax.transAxes, color="white",
            fontsize=14, fontweight="bold", va="top", ha="right",
            bbox=dict(boxstyle="round", facecolor="black", alpha=0.55))
        # Colormap bar on the right: Since the photo is already rendered with a color palette,
        # it is a reference bar indicating relative heat intensity rather than absolute temperature.
        thermal_sm = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(0, 1))
        cbar = self.fig.colorbar(thermal_sm, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Relative Heat Intensity", fontweight="bold", fontsize=13)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(["Low", "High"])
        cbar.ax.tick_params(labelsize=12)

    def _setup_section_panel(self):
        """Row 1, Col 2: Bead and Nugget Cross-section."""
        ax = self.ax_sec
        ax.set_title("2. Bead & Nugget Cross-section", fontweight="bold")
        ax.set_xlim(-6, 6); ax.set_ylim(-4, 4); ax.set_aspect("equal")
        ax.set_xlabel("Width (mm)"); ax.set_ylabel("Depth (mm)")
        thickness = float(self.frames["thickness_avg_mm"].mean())
        ax.axhline(0, color="black", lw=2)
        ax.axhline(-thickness, color="black", lw=2, ls="--")
        # Overlaps the nuggets of the two operation modes for direct size comparison
        self.sec_base = Ellipse((0, -thickness / 2), 0, 0, facecolor="none",
                                edgecolor=C_BASELINE_NUGGET, lw=2.5, ls="--", zorder=3)
        self.sec_opt = Ellipse((0, -thickness / 2), 0, 0, facecolor=C_OPTIMAL,
                               alpha=0.55, edgecolor="black", lw=1.5, zorder=2)
        ax.add_patch(self.sec_base); ax.add_patch(self.sec_opt)
        ax.legend(handles=[
            mpatches.Patch(facecolor=C_OPTIMAL, alpha=0.55, label="Optimal Nugget"),
            mpatches.Patch(facecolor="none", edgecolor=C_BASELINE_NUGGET, label="Baseline Nugget"),
        ], loc="upper right")

    def _setup_env_panel(self):
        """Row 1, Col 3: Environment and Quality Normalization Index."""
        ax = self.ax_env
        ax.set_title("3. Env & Quality Index (vs Standard)", fontweight="bold")
        self.env_bars = ax.bar(self.env_labels, [100] * 4, width=0.6, zorder=2)
        self._add_zone_bands(ax)
        # Top right: Legend indicating the meaning of background colors (Risk/Caution/Safe)
        ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="Risk"),
            mpatches.Patch(facecolor=C_CAUTION, label="Caution"),
            mpatches.Patch(facecolor=C_SAFE, label="Safe"),
        ], loc="upper right", framealpha=0.85)
        top = max(200.0, float(np.max(self.env_index)) * 1.2)
        ax.set_ylim(0, top)
        ax.set_ylabel("Normalized Index (%)")
        self.env_texts = [ax.text(i, 100, "", ha="center", fontsize=13,
                                  fontweight="bold", zorder=4) for i in range(4)]
        self.env_bands = [ax.plot([], [], "k-", lw=1.5, zorder=4)[0]
                          for _ in range(4)]

    def _setup_phase_panel(self):
        """Row 2, Col 1: Process Phase Diagram (Current x Time)."""
        ax = self.ax_phase
        ax.set_title("4. Process Phase Diagram (Current x Time)", fontweight="bold")
        I_grid = np.linspace(500, 4500, 240)
        t_grid = np.linspace(0.05, 1.65, 240)
        II, TT = np.meshgrid(I_grid, t_grid)
        QQ = II ** 2 * TT     # Heat input indicator with contact resistance normalized to 1
        q_lo = self.params["Q_min"] - self.params["Q_min_err"]
        q_hi = self.params["Q_min"] + self.params["Q_min_err"]
        # Classified as lack of fusion risk zone if heat input is below threshold
        zone = np.where(QQ < q_lo, 0, np.where(QQ < q_hi, 1, 2))
        ax.contourf(II, TT, zone, levels=[-0.5, 0.5, 1.5, 2.5],
                    colors=[C_RISK, C_CAUTION, C_SAFE], zorder=0)
        # Displays target heat input contour as a reference line
        cs = ax.contour(II, TT, QQ, levels=[self.Q_target], colors="black",
                        linestyles="--", linewidths=1.5, zorder=1)
        ax.clabel(cs, fmt={self.Q_target: "Target Q"}, fontsize=12)
        # Displays the distribution of 493 empirical cases in the background
        ax.scatter(self.repo.samples["avg_current_A"],
                   self.repo.samples["weld_time_s"],
                   s=10, c="#555555", alpha=0.25, zorder=2, label="493 Empirical Cases")
        self.phase_trail, = ax.plot([], [], color=C_OPTIMAL, lw=1.0,
                                    alpha=0.5, zorder=3)
        self.phase_base, = ax.plot([], [], "o", ms=12, color=C_BASELINE,
                                   mec="black", zorder=5, label="baseline")
        self.phase_opt, = ax.plot([], [], "o", ms=12, color=C_OPTIMAL,
                                  mec="black", zorder=6, label="optimal")
        ax.set_xlim(500, 4500); ax.set_ylim(0.05, 1.65)
        ax.set_xlabel("Current (A)"); ax.set_ylabel("Weld Time (s)")
        # Center left: Legend indicating the meaning of background colors (Risk/Caution/Safe).
        # Must be fixed with add_artist so it doesn't get erased when drawing the second legend below
        zone_legend_phase = ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="Risk"),
            mpatches.Patch(facecolor=C_CAUTION, label="Caution"),
            mpatches.Patch(facecolor=C_SAFE, label="Safe"),
        ],
                                      loc="center left", framealpha=0.85)
        ax.add_artist(zone_legend_phase)
        ax.legend(loc="upper right", framealpha=0.85)
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_heat_panel(self):
        """Row 2, Col 2: Effective Heat Input Trend.

        Background risk/caution/safe zones are based on the same lack of fusion threshold
        (Q_min +/- error) as the 2x1 process phase diagram. The two panels dealing with the same heat input Q
        must be colored with the same criteria for consistent interpretation.
        """
        ax = self.ax_heat
        ax.set_title("5. Effective Heat Input $Q_{eff}$ Trend", fontweight="bold")
        ax.set_xlim(0, self.n - 1)
        y_top = max(self.log["base_Q"].max(), self.log["opt_Q"].max()) * 1.1
        ax.set_ylim(0, y_top)
        # Displays lack of fusion risk/caution/safe zones in the background (same criteria as row 2 col 1)
        q_lo = self.params["Q_min"] - self.params["Q_min_err"]
        q_hi = self.params["Q_min"] + self.params["Q_min_err"]
        ax.axhspan(0, q_lo, color=C_RISK, zorder=0)
        ax.axhspan(q_lo, q_hi, color=C_CAUTION, zorder=0)
        ax.axhspan(q_hi, y_top, color=C_SAFE, zorder=0)
        ax.axhline(self.Q_target, color="black", ls="--", lw=1.5, zorder=2,
                   label="Target $Q$")
        self.heat_base, = ax.plot([], [], color=C_BASELINE, lw=1.5,
                                  label="baseline")
        self.heat_opt, = ax.plot([], [], color=C_OPTIMAL, lw=2.0,
                                 label="optimal")
        ax.set_xlabel("Frame (Sample Seq)"); ax.set_ylabel("Heat Input Indicator")
        # Center right: Legend indicating the meaning of background colors (Risk/Caution/Safe)
        zone_legend_heat = ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="Risk"),
            mpatches.Patch(facecolor=C_CAUTION, label="Caution"),
            mpatches.Patch(facecolor=C_SAFE, label="Safe"),
        ], loc="center right", framealpha=0.85)
        ax.add_artist(zone_legend_heat)
        ax.legend(loc="lower right")
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_machine_panel(self):
        """Row 2, Col 3: Machine Control Variable Normalization Index."""
        ax = self.ax_mach
        ax.set_title("6. Machine Variable Index (vs Standard)", fontweight="bold")
        self.mach_bars = ax.bar(self.mach_labels, [100] * 5, width=0.6, zorder=2)
        self._add_zone_bands(ax)
        # Top left: Legend indicating the meaning of background colors (Risk/Caution/Safe)
        ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="Risk"),
            mpatches.Patch(facecolor=C_CAUTION, label="Caution"),
            mpatches.Patch(facecolor=C_SAFE, label="Safe"),
        ],
                  loc="upper right", framealpha=0.85)
        top = max(200.0, float(np.max(self.mach_index)) * 1.2)
        ax.set_ylim(0, top)
        ax.set_ylabel("Normalized Index (%)")
        ax.tick_params(axis="x", labelsize=15)
        self.mach_texts = [ax.text(i, 100, "", ha="center", fontsize=13,
                                   fontweight="bold", zorder=4) for i in range(5)]
        self.mach_bands = [ax.plot([], [], "k-", lw=1.5, zorder=4)[0]
                           for _ in range(5)]

    def _setup_fit_panel(self):
        """Row 3, Col 1: Fitting Function and Real-time Operating Point."""
        ax = self.ax_fit
        ax.set_title("7. Real-time Operating Point on Fitting Curve", fontweight="bold")
        q_max = max(self.log["base_Q"].max(), self.log["opt_Q"].max()) * 1.15
        q_axis = np.linspace(1e4, q_max, 400)
        # Left axis: Lack of fusion probability curve and empirical interval statistics
        ax.plot(q_axis, bad_probability_model(
            q_axis, self.params["Q_min"], self.params["k_bad"]),
            color=FG_SAFE, lw=2, label="Lack of Fusion Prob $p_{bad}(Q)$")
        b = self.repo.binned
        ax.errorbar(b["heat_mean"], b["bad_rate"], yerr=b["bad_rate_sem"],
                    fmt="o", ms=5, color=FG_SAFE, ecolor="#d62728",
                    capsize=3, alpha=0.8, label="Empirical Interval Avg +/- SEM")
        ax.set_xlim(0, q_max); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("Heat Input Indicator $Q$"); ax.set_ylabel("Lack of Fusion Prob")
        # Right axis: Nugget growth curve
        ax2 = ax.twinx()
        ax2.plot(q_axis, nugget_growth_model(
            q_axis, self.params["D0"], self.params["Dmax"], self.params["tau"]),
            color="#9467bd", lw=2, ls="-.", label="Nugget Diameter $D(Q)$")
        # Match the right axis label and tick colors to the purple of the nugget diameter curve
        ax2.set_ylabel("Nugget Diameter (mm)", color="#9467bd")
        ax2.tick_params(axis="y", labelcolor="#9467bd")
        ax2.set_ylim(2.5, self.params["Dmax"] * 1.15)
        self.fit_base, = ax.plot([], [], "o", ms=12, color=C_BASELINE,
                                 mec="black", zorder=5)
        self.fit_opt, = ax.plot([], [], "o", ms=12, color=C_OPTIMAL,
                                mec="black", zorder=6)
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="upper right")
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_mv_panel(self):
        """Row 3, Col 2: Manipulated Variable Correction Trajectory.

        Weld time has small variation (seconds), while current and pressure have relatively large variations.
        Thus, weld time is placed on the left axis, and current/pressure on the right axis
        so they do not compromise each other's visibility.
        """
        ax = self.ax_mv
        ax.set_title("8. MV Real-time Correction Trajectory", fontweight="bold")
        ax.set_xlim(0, self.n - 1)

        # Left axis: Weld time correction ratio
        t_ratio = self.mv_ratio["time"]
        t_lo = min(0.0, float(t_ratio.min()) * 0.9)
        t_hi = float(t_ratio.max()) * 1.15
        ax.set_ylim(t_lo, t_hi)
        ax.axhline(100, color="#2ca02c", ls="--", lw=1.2, zorder=1)
        ax.set_xlabel("Frame (Sample Seq)")
        ax.set_ylabel("Weld Time Ratio (%)", color="#2ca02c")
        ax.tick_params(axis="y", labelcolor="#2ca02c")
        self.mv_lines = {}
        self.mv_lines["time"], = ax.plot([], [], color="#2ca02c", lw=1.8,
                                         label="Time (Left Axis)")

        # Right axis: Current & pressure correction ratio
        ax2 = ax.twinx()
        cp_ratio = np.concatenate([self.mv_ratio["current"], self.mv_ratio["pressure"]])
        cp_lo = min(0.0, float(cp_ratio.min()) * 0.9)
        cp_hi = float(cp_ratio.max()) * 1.15
        ax2.set_ylim(cp_lo, cp_hi)
        ax2.axhline(100, color="black", ls=":", lw=1.2, zorder=1)
        ax2.set_ylabel("Current/Pressure Ratio (%)")
        for key, color, label in [("current", "#d62728", "Current (Right Axis)"),
                                  ("pressure", "#ff7f0e", "Pressure (Right Axis)")]:
            self.mv_lines[key], = ax2.plot([], [], color=color, lw=1.8, label=label)
        self.ax_mv2 = ax2

        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc="upper right", ncol=1)
        ax.grid(True, ls=":", alpha=0.4)

    def _setup_defect_panel(self):
        """Row 3, Col 3: Total Defect Probability and Guardrail Performance."""
        ax = self.ax_def
        ax.set_title("9. Total Defect Probability & Guardrail", fontweight="bold")
        ax.set_xlim(0, self.n - 1); ax.set_ylim(0, 105)
        ax.axhspan(0, 20, color=C_SAFE, zorder=0)
        ax.axhspan(20, 50, color=C_CAUTION, zorder=0)
        ax.axhspan(50, 105, color=C_RISK, zorder=0)
        self.def_base, = ax.plot([], [], color=C_BASELINE, lw=1.5,
                                 label="baseline")
        self.def_opt, = ax.plot([], [], color=C_OPTIMAL, lw=2.0,
                                label="optimal")
        self.def_text = ax.text(0.98, 0.95, "", transform=ax.transAxes,
                                ha="right", va="top", fontsize=18,
                                fontweight="bold",
                                bbox=dict(boxstyle="round", facecolor="white",
                                          alpha=0.85))
        ax.set_xlabel("Frame (Sample Seq)"); ax.set_ylabel("Defect Probability (%)")
        # Center right: Legend indicating the meaning of background colors (Risk/Caution/Safe).
        # Must be fixed with add_artist so it doesn't get erased when drawing the baseline/optimal legend below
        zone_legend_def = ax.legend(handles=[
            mpatches.Patch(facecolor=C_RISK, label="Risk"),
            mpatches.Patch(facecolor=C_CAUTION, label="Caution"),
            mpatches.Patch(facecolor=C_SAFE, label="Safe"),
        ],
                                    loc="center right", framealpha=0.85)
        ax.add_artist(zone_legend_def)
        # Move slightly (+10x, -3y) from the default top-left position (approx x=0, y=105)
        # based on data coordinates
        ax.legend(loc="upper left", bbox_to_anchor=(10, 102),
                  bbox_transform=ax.transData)
        ax.grid(True, ls=":", alpha=0.4)

    # ---------- Frame Update ----------
    def _update(self, i):
        row = self.log.iloc[i]
        past = slice(0, i + 1)
        x_axis = np.arange(i + 1)

        # 1. IR Image
        self.img_artist.set_data(self.images[i])
        self.img_label.set_text(
            "Sample #%d  |  %s\nElectrode Angle %.0f°"
            % (row["sample_id"], row["category"], row["angle_deg"]))
        self.img_number_label.set_text("Image %03d / %03d" % (i + 1, self.n))

        # 2. Nugget Cross-section (approximate diameter as width, 60% as height)
        for patch, d in [(self.sec_base, row["base_nugget"]),
                         (self.sec_opt, row["opt_nugget"])]:
            patch.width, patch.height = d, d * 0.6

        # 3. Env & Quality Index
        env_now = self.env_index[:, i]
        env_raw = [T_REFERENCE_C + row["temp_dev_true"], row["tilt_true"],
                   row["opt_nugget"], row["opt_tensile"]]
        env_fmt = ["%.1f°", "%.1f°", "%.2fmm", "%.0fN"]
        for k, bar in enumerate(self.env_bars):
            bar.set_height(env_now[k])
            bar.set_color(self._state_color(env_now[k]))
            self.env_texts[k].set_text(env_fmt[k] % env_raw[k])
            self.env_texts[k].set_position((k, env_now[k] * 1.02))
            # Displays the variation range observed so far as vertical lines
            self.env_bands[k].set_data(
                [k, k], [self.env_index[k, past].min(),
                         self.env_index[k, past].max()])

        # 4. Process Phase Diagram
        self.phase_base.set_data([row["base_current"]], [row["base_time"]])
        self.phase_opt.set_data([row["opt_current"]], [row["opt_time"]])
        self.phase_trail.set_data(self.log["opt_current"].values[past],
                                  self.log["opt_time"].values[past])

        # 5. Effective Heat Input
        self.heat_base.set_data(x_axis, self.log["base_Q"].values[past])
        self.heat_opt.set_data(x_axis, self.log["opt_Q"].values[past])

        # 6. Machine Control Variable Index
        mach_now = self.mach_index[:, i]
        mach_raw = [row["opt_current"], self.V_opt[i], row["opt_time"],
                    row["opt_pressure"], row["opt_Q"]]
        mach_fmt = ["%.0fA", "%.2fV", "%.2fs", "%.0fpsi", "%.1e"]
        for k, bar in enumerate(self.mach_bars):
            bar.set_height(mach_now[k])
            bar.set_color(self._state_color(mach_now[k]))
            self.mach_texts[k].set_text(mach_fmt[k] % mach_raw[k])
            self.mach_texts[k].set_position((k, mach_now[k] * 1.02))
            self.mach_bands[k].set_data(
                [k, k], [self.mach_index[k, past].min(),
                         self.mach_index[k, past].max()])

        # 7. Operating point on fitting curve
        self.fit_base.set_data([row["base_Q"]], [row["base_p_bad"]])
        self.fit_opt.set_data([row["opt_Q"]], [row["opt_p_bad"]])

        # 8. MV Correction Ratio (baseline set to 100%, uses pre-calculated values)
        for key in ("current", "time", "pressure"):
            self.mv_lines[key].set_data(x_axis, self.mv_ratio[key][past])

        # 9. Total Defect Probability
        self.def_base.set_data(x_axis, self.log["base_p_total"].values[past] * 100)
        self.def_opt.set_data(x_axis, self.log["opt_p_total"].values[past] * 100)
        self.def_text.set_text(
            "baseline %.1f%%\noptimal  %.1f%%"
            % (row["base_p_total"] * 100, row["opt_p_total"] * 100))

    def render(self, gif_path, fps=8):
        self._setup()
        anim = animation.FuncAnimation(self.fig, self._update,
                                       frames=self.n, interval=1000 // fps,
                                       blit=False)
        anim.save(gif_path, writer="pillow", fps=fps)
        plt.close(self.fig)
        return anim


# --- Execution ---------------------------------------------------------
plt.rcParams["animation.embed_limit"] = 200

renderer = DashboardRenderer(repo, frames, log, params, Q_TARGET)
gif_path = os.path.join(FIGURE_DIR, "3x3_weld_dashboard_Eng.gif")
anim = renderer.render(gif_path)

print("[Completed] Saved %d-frame dashboard: %s" % (len(log), gif_path))
display(HTML(anim.to_jshtml()))